# 06b Ablations, alerts, extreme-life holdout
Phase 6b. Descriptive and exploratory only: nothing here is a gate result, nothing is tuned on test cells, no nested tuning is run. The frozen gate, module and Phase 5b verdict are hash-checked at the start and the end. Writes only to `outputs/phase6b`.

## 1. Guards

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import sys, time, json, gc, math, hashlib, inspect, re, warnings, ctypes, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd, psutil
from scipy.signal import lfilter
from scipy.stats import rankdata, spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
T_START = time.time()

def keep_awake(on):
    try:
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000001 if on else 0x80000000)
    except Exception as e:
        print("keep-awake unavailable:", e)
keep_awake(True)

# ---- root and paths (ROOT = parent of notebooks/)
_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "notebooks").is_dir() and (p / "data").is_dir()), None)
if ROOT is None:
    raise RuntimeError(f"cannot resolve ROOT (parent of notebooks/) from {_cwd}")
PROC = ROOT / "data/processed"; SPL = PROC / "splits"; P3 = ROOT / "outputs/phase3"; OUT4 = ROOT / "outputs/phase4"
OUT5B = ROOT / "outputs/phase5b"; SRC = ROOT / "src"
OUT = ROOT / "outputs/phase6b"; FIG = OUT / "figures"; CACHE = OUT / "cache"
for _p in (OUT, FIG, CACHE):
    _p.mkdir(parents=True, exist_ok=True)
P_GATE = OUT4 / "week2_gate.json"; P_MODULE = SRC / "soh_deep_v2.py"; P_VERDICT = OUT5B / "gate_verdict.json"
P_TUNING = OUT5B / "tuning_log_5b.csv"; P_DEEPRES = OUT5B / "deep_results.csv"; P_CACHE5B = OUT5B / "cache"

# ---- frozen facts
GATE_SHA = "40df2fb82096e994f8bf7b8084433bb5ca148d3fa92c780461f68c0c4247d901"
MODULE_SHA_EXPECTED = "fc53e37d506ebe51f6e4d71819c4b047de5bc38456669a160183531fbefebea6"
VERDICT_SHA = "2f59f82a7c7d8940636b55d7fd3397760d4486569d95575c802dbd02df69b1bb"
EXPECTED_ROWS, EXPECTED_CELLS = 55963, 84
NB_VERSION = "06b-v1"

# ---- checks (PASS / WARN / FAIL; FAIL raises)
CHECKS, WRITTEN, INTERVALS = [], [], {}
def dump_checks(partial=False, extra=None):
    n = lambda s: sum(c["level"] == s for c in CHECKS)
    rep = dict(phase="6b", created=dt.datetime.now().isoformat(timespec="seconds"), root=str(ROOT), partial=partial,
               n_pass=n("PASS"), n_warn=n("WARN"), n_fail=n("FAIL"), intervals=INTERVALS, n_intervals=int(sum(INTERVALS.values())), checks=CHECKS)
    if extra: rep.update(extra)
    (OUT / "verification_report.json").write_text(json.dumps(rep, indent=2, default=str), encoding="utf-8")
def chk(level, name, detail=""):
    CHECKS.append(dict(level=level, name=name, detail=str(detail)))
    print(f"[{level}] {name}" + (f" | {detail}" if str(detail) else ""))
    if level == "FAIL":
        dump_checks(partial=True)
        raise AssertionError(f"{name}: {detail}")
def need(cond, name, detail="", level="FAIL"):
    chk("PASS" if cond else level, name, detail)
def require_cols(df, cols, name):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise KeyError(f"{name}: missing column(s) {miss}; found {list(df.columns)}")
def add_int(section, n=1):
    INTERVALS[section] = INTERVALS.get(section, 0) + int(n)

# ---- guarded writes: nothing is ever written outside outputs/phase6b
def _guard(p):
    p = Path(p).resolve()
    if OUT.resolve() not in p.parents:
        raise RuntimeError(f"refusing to write outside {OUT}: {p}")
    return p
def save_csv(df, name):
    p = _guard(OUT / name); df.to_csv(p, index=False); WRITTEN.append(p); return p
def save_text(text, name):
    p = _guard(OUT / name); p.write_text(text, encoding="utf-8"); WRITTEN.append(p); return p
def save_json(obj, name):
    p = _guard(OUT / name); p.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8"); WRITTEN.append(p); return p
def save_parquet(df, path):
    p = _guard(path); df.to_parquet(p); return p
def save_npz(path, **kw):
    p = _guard(path); np.savez_compressed(p, **kw); return p
def save_npy(path, arr):
    p = _guard(path); np.save(p, arr); return p

def ram_report(tag=""):
    av = psutil.virtual_memory().available / 1e9
    procs = []
    for p in psutil.process_iter(["name", "memory_info"]):
        try: procs.append((p.info["memory_info"].rss / 1e9, p.info["name"]))
        except Exception: pass
    procs.sort(reverse=True)
    print(f"RAM available {av:.1f} GB {tag}; top 8:", ", ".join(f"{n} {g:.1f}GB" for g, n in procs[:8]))
    if av < 2.0:
        chk("WARN", "RAM below 2 GB", f"{av:.1f} GB available; close memory hogs")
    return av
ram_report("(start)")

# ---- frozen-file hashes (start)
def sha_raw(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def sha_lf(p): return hashlib.sha256(Path(p).read_bytes().replace(b"\r\n", b"\n")).hexdigest()
for _p in (P_GATE, P_MODULE, P_VERDICT, P_TUNING, P_DEEPRES):
    need(_p.exists(), f"input exists: {_p.relative_to(ROOT)}")
def hash_state():
    return dict(gate=dict(sha=sha_raw(P_GATE), exp=GATE_SHA, mtime=P_GATE.stat().st_mtime_ns),
                module=dict(sha=sha_lf(P_MODULE), exp=MODULE_SHA_EXPECTED, mtime=P_MODULE.stat().st_mtime_ns),
                verdict=dict(sha=sha_raw(P_VERDICT), exp=VERDICT_SHA, mtime=P_VERDICT.stat().st_mtime_ns))
HASH0 = hash_state()
for _k, _v in HASH0.items():
    need(_v["sha"] == _v["exp"], f"sha256 matches frozen value at start: {_k}", _v["sha"][:16] + "...")
MODULE_SHA = HASH0["module"]["sha"]
_V = json.loads(P_VERDICT.read_text())
print("frozen verdict (read only, never recomputed): headline W300 PASS =", _V.get("headline_PASS"), "| companion W240 PASS =", _V.get("companion_PASS"))
need(_V.get("headline_PASS") is False and _V.get("companion_PASS") is True, "verdict file reads W300 FAIL / W240 PASS", level="WARN")

# ---- module API (public names only; module is frozen)
sys.path.insert(0, str(SRC))
import soh_common as sc
import soh_deep_v2 as sd
for _n in ("train_members", "fit_scalers", "predict", "sub_idx", "count_params", "make_single", "make_inputs"):
    need(hasattr(sd, _n), f"soh_deep_v2.{_n} present")
def _sig(o):
    try: return str(inspect.signature(o))
    except Exception as e: return f"(unavailable: {e})"
print("public names:", [n for n in dir(sd) if not n.startswith("_")])
for _n in ("make_single", "make_batched", "train_members"):
    print(f"  {_n}{_sig(getattr(sd, _n))}")
_SRC = inspect.getsource(sd)
CFG_KEYS = sorted(set(re.findall(r'cfg\[\s*["\'](\w+)["\']\s*\]', _SRC)))
print("cfg keys used by the module:", CFG_KEYS)
# size / width arguments: the module exposes none unless a parameter name below appears on the model constructors
_size_params = [p for f in (sd.make_single, sd.make_batched) for p in inspect.signature(f).parameters if p.lower() in ("width", "mult", "c", "scale", "hidden", "channels", "size")]
SIZE_EXPOSED = bool(_size_params)
print("size-like parameters on make_single/make_batched:", _size_params or "none")

# ---- CUDA, fp32, TF32 off, deterministic
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try: torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e: print("deterministic algorithms flag unavailable:", e)
need(torch.cuda.is_available(), "CUDA available (needed for the training sections)")
_pr = torch.cuda.get_device_properties(0)
GPU_INFO = f"{_pr.name} {_pr.total_memory / 1e9:.1f} GB VRAM, torch {torch.__version__}"
DEVICE = torch.device("cuda")
print("GPU:", GPU_INFO, "| TF32 matmul:", torch.backends.cuda.matmul.allow_tf32, "| cudnn deterministic:", torch.backends.cudnn.deterministic)
sc.assert_allowed(["q0"], roles=("primary",))
chk("PASS", "allow-list q0 (primary)")

RAM available 4.9 GB (start); top 8: Code.exe 0.9GB, python.exe 0.5GB, Code.exe 0.4GB, explorer.exe 0.4GB, chrome.exe 0.4GB, chrome.exe 0.3GB, MsMpEng.exe 0.3GB, Code.exe 0.3GB
[PASS] input exists: outputs\phase4\week2_gate.json
[PASS] input exists: src\soh_deep_v2.py
[PASS] input exists: outputs\phase5b\gate_verdict.json
[PASS] input exists: outputs\phase5b\tuning_log_5b.csv
[PASS] input exists: outputs\phase5b\deep_results.csv
[PASS] sha256 matches frozen value at start: gate | 40df2fb82096e994...
[PASS] sha256 matches frozen value at start: module | fc53e37d506ebe51...
[PASS] sha256 matches frozen value at start: verdict | 2f59f82a7c7d8940...
frozen verdict (read only, never recomputed): headline W300 PASS = False | companion W240 PASS = True
[PASS] verdict file reads W300 FAIL / W240 PASS
[PASS] soh_deep_v2.train_members present
[PASS] soh_deep_v2.fit_scalers present
[PASS] soh_deep_v2.predict present
[PASS] soh_deep_v2.sub_idx present
[PASS] soh_deep_v2.count_params present
[PASS]

In [2]:
SENS_CELLS = {"b1c8", "b1c10", "b1c12", "b1c13", "b1c22"}
STITCHED = [f"b1c{i}" for i in range(5)]
ADDED = {"b1c0": 662, "b1c1": 981, "b1c2": 1060, "b1c3": 208, "b1c4": 482}
WINS = ["W300", "W240"]; GRID_V = {"W300": 3.00, "W240": 2.40}
SEEDS = [42, 43, 44, 45, 46]; B_BOOT, SEED_BOOT = 5000, 42
TERT_CUTS = (493, 710); TERT_LAB = ["short", "mid", "long"]

_lp = [p for p in (PROC / "labels_table_v2.parquet", SPL / "labels_table_v2.parquet", P3 / "labels_table_v2.parquet") if p.exists()]
need(bool(_lp), "labels_table_v2.parquet found")
labels = pd.read_parquet(_lp[0])
require_cols(labels, ["row_id", "cell_id", "batch", "cycle_number", "q0", "soh_nom", "scored2"], "labels_table_v2")
need(len(labels) == 104971, "labels rows", len(labels))
need(bool((labels["row_id"].values == np.arange(len(labels))).all()), "row_id == position")
cycles, qdlin, t_knots, registry = sc.load_tables(root=ROOT, mmap=True)
need(qdlin.shape == (len(labels), 1000), "qdlin shape", qdlin.shape)
qref = np.load(PROC / "qref_f32.npy"); qref_ids = pd.read_csv(PROC / "qref_cell_ids.csv")
require_cols(qref_ids, ["row", "cell_id"], "qref_cell_ids.csv")
need(qref.shape == (129, 1000) and bool((qref_ids["row"].values == np.arange(len(qref_ids))).all()), "qref shape/rows", qref.shape)
QREF_ROW = dict(zip(qref_ids["cell_id"], qref_ids["row"]))

pD = pd.read_csv(SPL / "protocol_D.csv")
require_cols(pD, ["repeat", "outer_fold", "cell_id", "role"], "protocol_D.csv")
need(set(pD["role"]) <= {"train", "test"}, "split roles")
DEV_CELLS = sorted(pD["cell_id"].unique())
need(len(DEV_CELLS) == EXPECTED_CELLS, "84 development cells", len(DEV_CELLS))
need(sum(c.startswith("b1") for c in DEV_CELLS) == 41 and sum(c.startswith("b2") for c in DEV_CELLS) == 43, "41 batch-1 + 43 batch-2")
need(not any(c.startswith("b3") for c in DEV_CELLS) and not (set(DEV_CELLS) & SENS_CELLS), "no batch-3 / sensitivity cells in development")
REPS = sorted(int(r) for r in pD["repeat"].unique()); FOLDS = sorted(int(f) for f in pD["outer_fold"].unique())
need((len(REPS), len(FOLDS)) == (3, 6), "3 repeats x 6 outer folds", (REPS, FOLDS))
_t = pD[pD["role"] == "test"]
need(bool((_t.groupby(["repeat", "cell_id"]).size() == 1).all()) and bool((_t.groupby("repeat")["cell_id"].nunique() == 84).all()), "each dev cell in exactly one test set per repeat")
_n = _t.groupby(["repeat", "outer_fold"]).size()
need(len(_n) == 18 and _n.min() >= 13 and _n.max() <= 17, "18 outer sets with 13-17 test cells", _n.tolist())

_life = pd.read_csv(P3 / "kept_cells_groups_strata.csv")
require_cols(_life, ["cell_id", "life"], "kept_cells_groups_strata.csv")
LIFE_MAP = dict(zip(_life["cell_id"], _life["life"]))
def _find_policy_file():
    for p in [P3 / "kept_cells_groups_strata.csv", *sorted(P3.glob("*.csv")), *sorted(SPL.glob("*.csv"))]:
        if not p.exists(): continue
        try: h = pd.read_csv(p, nrows=2)
        except Exception: continue
        if {"cell_id", "policy_norm"} <= set(h.columns): return p
    return None
_pp = _find_policy_file()
need(_pp is not None, "policy_norm found in a Phase 3 groups file", "looked in outputs/phase3/*.csv and data/processed/splits/*.csv for columns cell_id + policy_norm")
_pol = pd.read_csv(_pp).drop_duplicates("cell_id").set_index("cell_id")["policy_norm"]
print("policy_norm source:", _pp.relative_to(ROOT))
POLICY = {c: str(_pol.get(c)) for c in DEV_CELLS}
need(all(v not in ("None", "nan") for v in POLICY.values()), "policy_norm known for all 84 development cells")
print("distinct policy_norm among the 84 development cells:", len(set(POLICY.values())))

D = labels[labels["scored2"].astype(bool) & labels["cell_id"].isin(DEV_CELLS)]
D = D.sort_values(["cell_id", "cycle_number", "row_id"], kind="stable").reset_index(drop=True)
del labels; gc.collect()
need(len(D) == EXPECTED_ROWS, "development rows", len(D))
need(not D.duplicated(["cell_id", "cycle_number"]).any(), "(cell, cycle) unique")
D["y"] = 100.0 * D["soh_nom"].astype(float)
ROWID = D["row_id"].values.astype(np.int64); CELLID = D["cell_id"].values
Y = D["y"].values.astype(np.float64); Q0 = D["q0"].values.astype(np.float64); CYC = D["cycle_number"].values.astype(np.float64)
need(not (np.isnan(Y).any() or np.isnan(Q0).any()), "no NaN in y / q0")
cells = D["cell_id"].unique(); NC, N, R = len(cells), len(D), len(REPS)
COUNTS = D.groupby("cell_id", sort=False).size().values; STARTS = np.r_[0, np.cumsum(COUNTS)[:-1]]
CELL_CODE = np.repeat(np.arange(NC), COUNTS)
need(list(cells) == sorted(cells) and COUNTS.sum() == N and bool((CELL_CODE == pd.factorize(D["cell_id"])[0]).all()), "cell blocks contiguous and sorted")
CELL_POS = {c: np.arange(s, s + k) for c, s, k in zip(cells, STARTS, COUNTS)}
CIDX = {c: i for i, c in enumerate(cells)}
BATCH = np.array([int(c[1]) for c in cells]); IS_ST = np.isin(cells, STITCHED)
LIFE = np.array([LIFE_MAP[c] for c in cells], dtype=float); LIFE_ROW = LIFE[CELL_CODE]
TERT = np.digitize(LIFE, TERT_CUTS)
print("life tertile sizes (cuts 493/710):", {TERT_LAB[k]: int((TERT == k).sum()) for k in range(3)}, "| life range", LIFE.min(), LIFE.max())
need(min((TERT == k).sum() for k in range(3)) >= 20, "tertile groups not tiny", level="WARN")
need(not (set(cells) & SENS_CELLS) and not any(str(c).startswith("b3") for c in cells), "cell arrays contain no batch-3 / sensitivity cells")
ROW_INDEX = pd.Index(ROWID)
HALF = (CYC >= 0.5 * LIFE_ROW).astype(int); TENTH = np.clip(np.floor(10.0 * CYC / LIFE_ROW), 0, 9).astype(int)

FOLD_CELL = np.full((R, NC), -1, dtype=int)
for _ri, _r in enumerate(REPS):
    for _row in _t[_t["repeat"] == _r].itertuples():
        FOLD_CELL[_ri, CIDX[_row.cell_id]] = int(_row.outer_fold)
need(bool((FOLD_CELL >= 0).all()), "fold map complete")
FOLD_ROW = FOLD_CELL[:, CELL_CODE]

def eval_pos(cs):
    return np.concatenate([CELL_POS[c] for c in sorted(cs)])
def build_sets():
    S = []
    for r, f in sorted(pD[["repeat", "outer_fold"]].drop_duplicates().itertuples(index=False, name=None)):
        g = pD[(pD["repeat"] == r) & (pD["outer_fold"] == f)]
        tr = sorted(g[g["role"] == "train"]["cell_id"]); te = sorted(g[g["role"] == "test"]["cell_id"])
        need(not set(tr) & set(te) and len(tr) + len(te) == 84, f"train/test disjoint and complete ({r},{f})")
        need(not any(c.startswith("b3") or c in SENS_CELLS for c in tr + te), f"no b3/sensitivity ({r},{f})")
        S.append(dict(repeat=int(r), outer_fold=int(f), train=tr, test=te))
    return S
SETS = build_sets()

# ---- schema of the saved predictions
import pyarrow.parquet as pq
_PRED_COLS = ["window", "model", "repeat", "outer_fold", "seed", "row_id", "cell_id", "pred_pp"]
_cache_files = sorted(p for p in P_CACHE5B.glob("final_*.parquet") if p.is_file())     # top level only; cache/smoke ignored
need(len(_cache_files) > 0, "Phase 5b cache: final_*.parquet files (top level)", f"{len(_cache_files)} files")
for _p in _cache_files:
    _miss = [c for c in _PRED_COLS if c not in pq.read_schema(_p).names]
    need(not _miss, f"schema {_p.name}", f"missing: {_miss}" if _miss else "")
_tl = pd.read_csv(P_TUNING); require_cols(_tl, ["window", "model", "config", "repeat", "outer_fold", "inner_fold", "mae"], "tuning_log_5b.csv")
_dr = pd.read_csv(P_DEEPRES); require_cols(_dr, ["window", "model", "macro_mean"], "deep_results.csv")
chk("PASS", "schemas: tuning_log_5b.csv, deep_results.csv, final_*.parquet")
ram_report("(data)")

[PASS] labels_table_v2.parquet found
[PASS] labels rows | 104971
[PASS] row_id == position
[PASS] qdlin shape | (104971, 1000)
[PASS] qref shape/rows | (129, 1000)
[PASS] split roles
[PASS] 84 development cells | 84
[PASS] 41 batch-1 + 43 batch-2
[PASS] no batch-3 / sensitivity cells in development
[PASS] 3 repeats x 6 outer folds | ([0, 1, 2], [0, 1, 2, 3, 4, 5])
[PASS] each dev cell in exactly one test set per repeat
[PASS] 18 outer sets with 13-17 test cells | [14, 14, 14, 15, 14, 13, 14, 14, 13, 17, 13, 13, 14, 14, 14, 14, 14, 14]
[PASS] policy_norm found in a Phase 3 groups file | looked in outputs/phase3/*.csv and data/processed/splits/*.csv for columns cell_id + policy_norm
policy_norm source: outputs\phase3\kept_cells_groups_strata.csv
[PASS] policy_norm known for all 84 development cells
distinct policy_norm among the 84 development cells: 61
[PASS] development rows | 55963
[PASS] (cell, cycle) unique
[PASS] no NaN in y / q0
[PASS] cell blocks contiguous and sorted
life tertil

4.11308032

In [3]:
# ---- Phase 5b out-of-fold predictions -> seed-ensemble arrays (R, N); M0, M1p, B4r at W300 / W240 only
NEED_PM = [(w, m) for w in WINS for m in ("M0", "M1p", "B4r")]
_sig_cache = json.dumps(dict(files=[(p.name, p.stat().st_size, p.stat().st_mtime_ns) for p in _cache_files], rows=hashlib.sha256(ROWID.tobytes()).hexdigest(), reps=REPS, v=NB_VERSION))
_npz = CACHE / "ensemble_arrays_5b.npz"; PRED, META, _loaded = {}, {}, False
if _npz.exists():
    try:
        _z = np.load(_npz, allow_pickle=False)
        if str(_z["sig"]) == _sig_cache:
            META = json.loads(str(_z["meta"])); PRED = {tuple(k.split("|")): _z["P__" + k.replace("|", "__")] for k in META}; _loaded = True
            print(f"resumed ensemble arrays from {_npz.name}")
    except Exception as e:
        print("cache unreadable, rebuilding:", e)
if not _loaded:
    acc = {}
    for fp in _cache_files:
        df = pd.read_parquet(fp, columns=_PRED_COLS)
        df = df[pd.MultiIndex.from_arrays([df["window"].astype(str), df["model"].astype(str)]).isin(NEED_PM)]
        if df.empty: continue
        need(df.cell_id.isin(set(cells)).all(), f"{fp.name}: only development cells")
        pos = ROW_INDEX.get_indexer(df.row_id.values)
        need(bool((pos >= 0).all()), f"{fp.name}: every row_id is a development row")
        need(bool((cells[CELL_CODE[pos]] == df.cell_id.values).all()), f"{fp.name}: cell_id agrees with labels")
        need(not df.pred_pp.isna().any(), f"{fp.name}: no NaN predictions")
        pr = df.pred_pp.values.astype(float); fo = df.outer_fold.values
        for (w, m, r), idx in df.groupby(["window", "model", "repeat"], sort=False).indices.items():
            w, m, r = str(w), str(m), int(r)
            a = acc.setdefault((w, m, r), dict(sum=np.zeros(N), n=np.zeros(N), seeds=set(), fold_bad=0))
            p = pos[idx]
            a["sum"] += np.bincount(p, weights=pr[idx], minlength=N); a["n"] += np.bincount(p, minlength=N)
            a["seeds"].update(int(s) for s in np.unique(df.seed.values[idx]))
            a["fold_bad"] += int((fo[idx] != FOLD_ROW[REPS.index(r)][p]).sum())
        del df, pos, pr, fo
    for (w, m) in sorted({(k[0], k[1]) for k in acc}):
        arr = np.full((R, N), np.nan); okn, sb, fb = True, [], 0
        reps_here = sorted(k[2] for k in acc if k[:2] == (w, m))
        for r in reps_here:
            a = acc[(w, m, r)]; okn &= bool(np.all(a["n"] == len(a["seeds"]))); sb.append(sorted(a["seeds"])); fb += a["fold_bad"]
            arr[REPS.index(r)] = a["sum"] / np.maximum(a["n"], 1)
        PRED[(w, m)] = arr
        META["|".join((w, m))] = dict(repeats=reps_here, seeds=sb, every_row_has_all_seeds_once=okn, fold_mismatch=fb)
    save_npz(_npz, sig=np.array(_sig_cache), meta=np.array(json.dumps(META)), **{"P__" + "__".join(k): v for k, v in PRED.items()})
for (w, m) in NEED_PM:
    mk = META.get(f"{w}|{m}")
    need(mk is not None, f"present: {w} {m}")
    exp_seeds = SEEDS if m in ("M0", "M1p") else None
    seeds_ok = all(s == exp_seeds for s in mk["seeds"]) if exp_seeds else all(len(s) == 1 for s in mk["seeds"])
    need(mk["repeats"] == REPS and seeds_ok and mk["every_row_has_all_seeds_once"] and mk["fold_mismatch"] == 0, f"coverage {w} {m}: 3 repeats x {N:,} rows, seeds {'42-46' if exp_seeds else '1'}, folds match protocol D")
    need(PRED[(w, m)].shape == (R, N) and not np.isnan(PRED[(w, m)]).any(), f"84 cells / {N:,} rows, no NaN: {w} {m}")
ERR = {k: v - Y for k, v in PRED.items()}       # err = pred - y (positive = SOH over-estimated)

# ---- reproduction of deep_results.csv macro_mean within 1e-6, before the saved predictions are used
def seg_mean(x):
    return np.add.reduceat(x, STARTS, axis=-1) / COUNTS
REPRO = []
for (w, m), e in sorted(ERR.items()):
    mine = float(seg_mean(np.abs(e)).mean(0).mean()); ref = _dr[(_dr.window.astype(str) == w) & (_dr.model.astype(str) == m)].macro_mean.values
    need(len(ref) == 1, f"deep_results has {w} {m}", len(ref))
    REPRO.append(dict(window=w, model=m, recomputed=mine, deep_results=float(ref[0]), abs_diff=abs(mine - float(ref[0]))))
    need(abs(mine - float(ref[0])) <= 1e-6, f"reproduces deep_results macro_mean {w} {m}", f"{mine:.6f} vs {float(ref[0]):.6f}")
REPRO = pd.DataFrame(REPRO); save_csv(REPRO, "reproduction_check.csv")

[PASS] final_0484477492f5dfaf92bf.parquet: only development cells
[PASS] final_0484477492f5dfaf92bf.parquet: every row_id is a development row
[PASS] final_0484477492f5dfaf92bf.parquet: cell_id agrees with labels
[PASS] final_0484477492f5dfaf92bf.parquet: no NaN predictions
[PASS] final_0c73fa7634974d33a357.parquet: only development cells
[PASS] final_0c73fa7634974d33a357.parquet: every row_id is a development row
[PASS] final_0c73fa7634974d33a357.parquet: cell_id agrees with labels
[PASS] final_0c73fa7634974d33a357.parquet: no NaN predictions
[PASS] final_23e83b31829ec37a8f5b.parquet: only development cells
[PASS] final_23e83b31829ec37a8f5b.parquet: every row_id is a development row
[PASS] final_23e83b31829ec37a8f5b.parquet: cell_id agrees with labels
[PASS] final_23e83b31829ec37a8f5b.parquet: no NaN predictions
[PASS] final_2d766dbcb1222f92dbdb.parquet: only development cells
[PASS] final_2d766dbcb1222f92dbdb.parquet: every row_id is a development row
[PASS] final_2d766dbcb1222f92dbd

WindowsPath('C:/workstation/3RD YEAR/Battery-Health-Estimation/outputs/phase6b/reproduction_check.csv')

In [4]:
# ---- D69: fixed ablation / Phase 7 config = M0 (lr, wd) with the lowest pooled inner MAE per window (all rows of the tuning log)
def parse_m0(name):
    m = re.fullmatch(r"lr([0-9.eE+-]+)_wd([0-9.eE+-]+)", name)
    if m is None: raise ValueError(f"M0 config name does not parse: {name}")
    return float(m.group(1)), float(m.group(2))
def parse_leaf(name):
    m = re.fullmatch(r"leaf(\d+)", name)
    if m is None: raise ValueError(f"B4r config name does not parse: {name}")
    return int(m.group(1))
FROZEN_CFG, RANK_TABLES = {}, []
for w in WINS:
    g = _tl[(_tl.window == w) & (_tl.model == "M0")]
    need(len(g) > 0 and np.isfinite(g.mae).all(), f"M0 tuning rows for {w}", len(g))
    t = g.groupby("config")["mae"].agg(pooled_inner_mae="mean", n_rows="count").reset_index()
    t["lr"], t["wd"] = zip(*[parse_m0(c) for c in t.config])
    t = t.sort_values(["pooled_inner_mae", "lr", "wd"]).reset_index(drop=True); t.insert(0, "rank", np.arange(1, len(t) + 1)); t.insert(0, "window", w)
    RANK_TABLES.append(t); best = t.iloc[0]
    gb = _tl[(_tl.window == w) & (_tl.model == "B4r")]
    tb = gb.groupby("config")["mae"].agg(pooled_inner_mae="mean", n_rows="count").reset_index()
    tb["leaf"] = [parse_leaf(c) for c in tb.config]; tb = tb.sort_values(["pooled_inner_mae", "leaf"]).reset_index(drop=True)
    FROZEN_CFG[w] = dict(config=best.config, lr=float(best.lr), wd=float(best.wd), pooled_inner_mae=float(best.pooled_inner_mae),
                         b4r_leaf=int(tb.iloc[0].leaf), b4r_config=tb.iloc[0].config, b4r_pooled_inner_mae=float(tb.iloc[0].pooled_inner_mae))
    print(f"\nD69 ranking, {w}, M0 (pooled inner MAE over all {int(t.n_rows.iloc[0])} log rows per config):")
    print(t[["rank", "config", "lr", "wd", "pooled_inner_mae"]].to_string(index=False, float_format=lambda v: f"{v:.6f}"))
    print(f"  -> chosen {best.config}; B4r leaf by the same pooled rule: {tb.iloc[0].config} ({tb.iloc[0].pooled_inner_mae:.6f})")
RANK_DF = pd.concat(RANK_TABLES, ignore_index=True); save_csv(RANK_DF, "d69_ranking_M0.csv")
COMMON = dict(E=120, batch=256, seeds=SEEDS, stride=5, rows="cell-balanced, per-cell random offset (as 05b)", head="abs", huber_delta_pp=1.0, noise_sigma=0.02,
              clip=1.0, warmup_frac=0.05, nominal_ah=1.1, trees=300, tree_max_features=0.33, tree_seed=42)
FZ = dict(rule="D69: per window, the (lr, wd) of M0 with the lowest mean inner MAE pooled over all rows of outputs/phase5b/tuning_log_5b.csv for that window; B4r leaf by the same pooled rule",
          created=dt.datetime.now().isoformat(timespec="seconds"), source_tuning_log_sha256=sha_raw(P_TUNING), module_sha256=MODULE_SHA, gate_sha256=GATE_SHA,
          common=COMMON, windows=FROZEN_CFG)
_p = save_json(FZ, "frozen_config.json"); FZ_SHA = sha_raw(_p); save_text(FZ_SHA + "\n", "frozen_config.sha256.txt")
print("\nfrozen_config.json sha256:", FZ_SHA)
chk("PASS", "D69 frozen config written", f"{ {w: FROZEN_CFG[w]['config'] for w in WINS} } sha256 {FZ_SHA[:16]}...")

[PASS] M0 tuning rows for W300 | 432

D69 ranking, W300, M0 (pooled inner MAE over all 72 log rows per config):
 rank           config       lr       wd  pooled_inner_mae
    1 lr0.004_wd0.0001 0.004000 0.000100          0.897133
    2  lr0.004_wd0.001 0.004000 0.001000          0.900601
    3 lr0.002_wd0.0001 0.002000 0.000100          0.902514
    4 lr0.008_wd0.0001 0.008000 0.000100          0.907108
    5  lr0.002_wd0.001 0.002000 0.001000          0.910396
    6  lr0.008_wd0.001 0.008000 0.001000          0.912695
  -> chosen lr0.004_wd0.0001; B4r leaf by the same pooled rule: leaf3 (0.867370)
[PASS] M0 tuning rows for W240 | 432

D69 ranking, W240, M0 (pooled inner MAE over all 72 log rows per config):
 rank           config       lr       wd  pooled_inner_mae
    1  lr0.008_wd0.001 0.008000 0.001000          0.580805
    2 lr0.004_wd0.0001 0.004000 0.000100          0.585182
    3 lr0.008_wd0.0001 0.008000 0.000100          0.585199
    4  lr0.004_wd0.001 0.004000 0.001000      

## 2. No-training diagnostics from saved predictions
Windows W300 and W240; models M0, M1p, B4r; seed ensemble per row, mean over the 3 repeats per cell. err = pred - y (positive = SOH over-estimated). Bootstrap = cells resampled with replacement, 5000 draws, seed 42. Everything below is descriptive.

In [5]:
# ---- machinery (as 06a)
def cellgrp(vals, grp, K):
    S = np.zeros(NC * K); C = np.zeros(NC * K)
    for r in range(vals.shape[0]):
        g = grp if grp.ndim == 1 else grp[r]; ok = g >= 0; gid = CELL_CODE[ok] * K + g[ok]
        S += np.bincount(gid, weights=vals[r][ok], minlength=NC * K); C += np.bincount(gid, minlength=NC * K)
    with np.errstate(invalid="ignore", divide="ignore"):
        M = S / C
    return M.reshape(NC, K), C.reshape(NC, K), S.reshape(NC, K)
_WB = {}
def W_for(n, B=B_BOOT, seed=SEED_BOOT):
    if n not in _WB: _WB[n] = np.random.default_rng(seed).multinomial(n, np.full(n, 1.0 / n), size=B).astype(float)
    return _WB[n]
def wmean(W, M):
    ok = ~np.isnan(M); Z = np.where(ok, M, 0.0)
    with np.errstate(invalid="ignore", divide="ignore"):
        return (W @ Z) / (W @ ok.astype(float))
def ci95(draws):
    return np.nanpercentile(draws, [2.5, 97.5], axis=0)
W_ALL = W_for(NC)
_BI = {}
def boot_idx(n):
    if n not in _BI: _BI[n] = np.random.default_rng(SEED_BOOT).integers(0, n, (B_BOOT, n))
    return _BI[n]
def spearman_ci(x, y):
    rho = float(spearmanr(x, y).statistic); idx = boot_idx(len(x))
    rx = rankdata(x[idx], axis=1); ry = rankdata(y[idx], axis=1)
    rx = rx - rx.mean(1, keepdims=True); ry = ry - ry.mean(1, keepdims=True)
    den = np.sqrt((rx ** 2).sum(1) * (ry ** 2).sum(1)); r = np.where(den > 0, (rx * ry).sum(1) / np.where(den > 0, den, 1.0), np.nan)
    lo, hi = np.nanpercentile(r, [2.5, 97.5]); add_int("2a spearman CIs")
    return rho, float(lo), float(hi)

DIAG_COMBOS = [(w, m) for w in WINS for m in ("M0", "M1p", "B4r")]
CMX = {}
pc_rows = []
for (w, m) in DIAG_COMBOS:
    e = ERR[(w, m)]; bias_rc = seg_mean(e); mae_rc = seg_mean(np.abs(e)); mse_rc = seg_mean(e ** 2)
    sd_c = np.sqrt(np.maximum(mse_rc - bias_rc ** 2, 0.0)).mean(0); half, _, _ = cellgrp(e, HALF, 2)
    CMX[(w, m)] = dict(mae=mae_rc.mean(0), bias=bias_rc.mean(0), sd=sd_c, h1=half[:, 0], h2=half[:, 1])
    for c in range(NC):
        pc_rows.append(dict(window=w, model=m, cell_id=cells[c], batch=int(BATCH[c]), life=LIFE[c], stitched=bool(IS_ST[c]), mae=CMX[(w, m)]["mae"][c],
                            mean_signed_err=CMX[(w, m)]["bias"][c], within_cell_resid_sd=sd_c[c], signed_err_first_half=half[c, 0], signed_err_second_half=half[c, 1]))
PER_CELL = pd.DataFrame(pc_rows); save_csv(PER_CELL, "per_cell_bias.csv")

sp_rows, tert_rows = [], []
for (w, m) in DIAG_COMBOS:
    b = CMX[(w, m)]["bias"]
    for sname, mask in (("all84", np.ones(NC, bool)), ("excl_stitched", ~IS_ST)):
        rho, lo, hi = spearman_ci(LIFE[mask], b[mask])
        sp_rows.append(dict(window=w, model=m, subset=sname, n_cells=int(mask.sum()), spearman_bias_vs_life=rho, ci95_lo=lo, ci95_hi=hi))
        for k, lab in enumerate(TERT_LAB):
            mk = mask & (TERT == k)
            tert_rows.append(dict(window=w, model=m, subset=sname, life_tertile=lab, n_cells=int(mk.sum()), mean_signed_err=b[mk].mean(), macro_mae=CMX[(w, m)]["mae"][mk].mean(),
                                  share_cells_bias_pos=(b[mk] > 0).mean()))
SPEAR = pd.DataFrame(sp_rows); TERT_BIAS = pd.DataFrame(tert_rows)
save_csv(SPEAR, "bias_vs_life_spearman.csv"); save_csv(TERT_BIAS, "bias_by_life_tertile.csv")
print("Spearman correlation of per-cell bias with life (cell bootstrap CI, 5000 draws, seed 42):"); print(SPEAR.to_string(index=False))
print("\nBias by life tertile (cuts 493 / 710):"); print(TERT_BIAS.to_string(index=False))
print(f"\n{INTERVALS.get('2a spearman CIs', 0)} bootstrap intervals in 2a; descriptive.")

Spearman correlation of per-cell bias with life (cell bootstrap CI, 5000 draws, seed 42):
window model        subset  n_cells  spearman_bias_vs_life  ci95_lo  ci95_hi
  W300    M0         all84       84                 0.4991   0.2906   0.6693
  W300    M0 excl_stitched       79                 0.3990   0.1794   0.5917
  W300   M1p         all84       84                 0.4661   0.2480   0.6451
  W300   M1p excl_stitched       79                 0.3672   0.1411   0.5682
  W300   B4r         all84       84                 0.4973   0.2931   0.6639
  W300   B4r excl_stitched       79                 0.4117   0.1972   0.5929
  W240    M0         all84       84                 0.2081  -0.0114   0.4117
  W240    M0 excl_stitched       79                 0.1261  -0.0957   0.3474
  W240   M1p         all84       84                 0.2420   0.0174   0.4463
  W240   M1p excl_stitched       79                 0.1322  -0.0905   0.3505
  W240   B4r         all84       84                 0.3070   0.

In [6]:
# ---- 2b junction analysis, 5 stitched cells
J_ROWS, JUNC = [], {}
for cid in STITCHED:
    ci_ = CIDX[cid]; s, n = STARTS[ci_], COUNTS[ci_]; cyc, yv = CYC[s:s + n], Y[s:s + n]
    j0 = int(round(LIFE[ci_] - ADDED[cid])); dy, gap = np.diff(yv), np.diff(cyc)
    med = float(np.median(np.abs(dy[gap == 1]))) if (gap == 1).any() else np.nan
    win_ = (cyc[1:] >= j0 - 30) & (cyc[1:] <= j0 + 30) & (gap <= 3)
    need(bool(win_.any()), f"scored rows exist within +/-30 cycles of the approximate junction {cid}", j0)
    k = int(np.where(win_)[0][np.argmin(dy[win_])]); j1 = int(cyc[k + 1]); step = float(dy[k]); z = abs(step) / med if med and med > 0 else np.nan
    use_ref = bool(step < 0 and np.isfinite(z) and z >= 10)          # rule fixed in advance: take the refined cycle only for a clear negative outlier step
    JUNC[cid] = dict(j0=j0, j1=j1, step=step, z=z, used=j1 if use_ref else j0, rule="refined" if use_ref else "approx")
    need(abs(j1 - j0) <= 2 and use_ref, f"junction {cid}: approx {j0}, largest negative true step at {j1} ({step:+.3f} pp, {z:.1f}x median step)", level="WARN")
def _win(cyc, a, b, incl_b=False):
    return (cyc >= a) & ((cyc <= b) if incl_b else (cyc < b))
for (w, m) in [(w, m) for w in WINS for m in ("M0", "M1p", "B4r")]:
    Pm = PRED[(w, m)].mean(0); em = Pm - Y
    for cid in STITCHED:
        ci_ = CIDX[cid]; s, n = STARTS[ci_], COUNTS[ci_]; cyc = CYC[s:s + n]; ju = JUNC[cid]["used"]
        yv, pv, ev = Y[s:s + n], Pm[s:s + n], em[s:s + n]
        b20, a20 = _win(cyc, ju - 20, ju), _win(cyc, ju, ju + 20); b100, a100 = _win(cyc, ju - 100, ju), _win(cyc, ju, ju + 100, True); pre, post = cyc < ju, cyc >= ju
        mean_ = lambda v, k: float(v[k].mean()) if k.any() else np.nan
        true_step = mean_(yv, a20) - mean_(yv, b20); pred_step = mean_(pv, a20) - mean_(pv, b20)
        allm, prem, postm = float(ev.mean()), mean_(ev, pre), mean_(ev, post)
        J_ROWS.append(dict(window=w, model=m, cell_id=cid, life=LIFE[ci_], junction_approx=JUNC[cid]["j0"], junction_refined=JUNC[cid]["j1"], junction_used=ju, junction_rule=JUNC[cid]["rule"],
                           true_step_pp=true_step, pred_step_pp=pred_step, step_ratio=pred_step / true_step if abs(true_step) > 0.05 else np.nan,
                           err_100_before=mean_(ev, b100), err_100_after=mean_(ev, a100), change_in_err=mean_(ev, a100) - mean_(ev, b100),
                           mean_err_pre_junction=prem, mean_err_post_junction=postm, mean_err_all=allm,
                           frac_of_mean_err_before=prem / allm if abs(allm) > 1e-9 else np.nan,
                           mae_pre_junction=mean_(np.abs(ev), pre), mae_all=float(np.abs(ev).mean()), share_rows_before=float(pre.mean())))
JUNC_DF = pd.DataFrame(J_ROWS); save_csv(JUNC_DF, "junction_analysis.csv")
save_csv(pd.DataFrame([dict(cell_id=c, **v) for c, v in JUNC.items()]), "junction_locations.csv")
print("Junction analysis (step = mean over the 20 cycles after minus the 20 before; err windows are 100 cycles; pre-junction = all rows before the junction):")
print(JUNC_DF[JUNC_DF.model.isin(["M0", "B4r"])][["window", "model", "cell_id", "junction_used", "true_step_pp", "pred_step_pp", "err_100_before", "err_100_after", "mean_err_pre_junction", "mean_err_all", "frac_of_mean_err_before"]].to_string(index=False))

[PASS] scored rows exist within +/-30 cycles of the approximate junction b1c0 | 1190
[WARN] junction b1c0: approx 1190, largest negative true step at 1190 (-0.390 pp, 6.9x median step)
[PASS] scored rows exist within +/-30 cycles of the approximate junction b1c1 | 1179
[PASS] junction b1c1: approx 1179, largest negative true step at 1179 (-1.244 pp, 69.4x median step)
[PASS] scored rows exist within +/-30 cycles of the approximate junction b1c2 | 1177
[PASS] junction b1c2: approx 1177, largest negative true step at 1177 (-1.248 pp, 76.5x median step)
[PASS] scored rows exist within +/-30 cycles of the approximate junction b1c3 | 1226
[WARN] junction b1c3: approx 1226, largest negative true step at 1252 (-0.153 pp, 12.3x median step)
[PASS] scored rows exist within +/-30 cycles of the approximate junction b1c4 | 1227
[PASS] junction b1c4: approx 1227, largest negative true step at 1227 (-1.316 pp, 90.3x median step)
Junction analysis (step = mean over the 20 cycles after minus the 20 be

In [7]:
# ---- 2c M0 + B4r average (EXPLORATORY; not a gate result, not pre-registered)
AVG = {w: 0.5 * (PRED[(w, "M0")] + PRED[(w, "B4r")]) for w in WINS}
AVG_CM = {w: seg_mean(np.abs(AVG[w] - Y)).mean(0) for w in WINS}
SUBS = {"all84": np.ones(NC, bool), "excl_stitched": ~IS_ST, "life_short": TERT == 0, "life_mid": TERT == 1, "life_long": TERT == 2}
avg_rows = []
for w in WINS:
    for sname, mask in SUBS.items():
        Wm = W_for(int(mask.sum()))
        for other in ("M0", "B4r"):
            d = (AVG_CM[w] - CMX[(w, other)]["mae"])[mask]; lo, hi = ci95(wmean(Wm, d)); add_int("2c paired intervals")
            avg_rows.append(dict(window=w, subset=sname, contrast=f"avg(M0,B4r) - {other}", n_cells=int(mask.sum()), avg_macro_mae=AVG_CM[w][mask].mean(),
                                 other_macro_mae=CMX[(w, other)]["mae"][mask].mean(), diff=d.mean(), ci95_lo=lo, ci95_hi=hi))
AVG_DF = pd.DataFrame(avg_rows); AVG_DF["label"] = "EXPLORATORY"; save_csv(AVG_DF, "ensemble_avg_M0_B4r.csv")
print("EXPLORATORY: per-row average of the M0 seed-ensemble and the B4r prediction (negative diff = average better). Paired cluster bootstrap, 5000 draws.")
print(AVG_DF.drop(columns="label").to_string(index=False)); print(f"{INTERVALS.get('2c paired intervals', 0)} intervals in 2c.")

EXPLORATORY: per-row average of the M0 seed-ensemble and the B4r prediction (negative diff = average better). Paired cluster bootstrap, 5000 draws.
window        subset          contrast  n_cells  avg_macro_mae  other_macro_mae    diff  ci95_lo  ci95_hi
  W300         all84  avg(M0,B4r) - M0       84         0.7784           0.8360 -0.0576  -0.1002  -0.0140
  W300         all84 avg(M0,B4r) - B4r       84         0.7784           0.8180 -0.0396  -0.0803   0.0014
  W300 excl_stitched  avg(M0,B4r) - M0       79         0.6712           0.7014 -0.0303  -0.0689   0.0104
  W300 excl_stitched avg(M0,B4r) - B4r       79         0.6712           0.7404 -0.0693  -0.1046  -0.0373
  W300    life_short  avg(M0,B4r) - M0       28         0.6945           0.7008 -0.0063  -0.0775   0.0522
  W300    life_short avg(M0,B4r) - B4r       28         0.6945           0.7855 -0.0910  -0.1346  -0.0492
  W300      life_mid  avg(M0,B4r) - M0       28         0.7512           0.7599 -0.0086  -0.0630   0.0657
  W3

In [8]:
# ---- 2d one-sided conformal lower bounds and alert rules
LEVEL, NBINS, ALPHAS, EARLY_PCT = 0.90, 6, [0.2, 0.3, 0.5], -5.0
AL_COMBOS = [(w, m) for w in WINS for m in ("M0", "B4r")]
def kth_smallest(res, level):
    n = res.size; k = min(int(math.ceil((n + 1) * level)), n)
    return float(np.partition(res, k - 1)[k - 1])
def onesided_q(P):
    # q = 90% finite-sample quantile of the signed error (pred - y); bins on predicted SOH; calibration = the other 5 folds of the same repeat
    e = P - Y; Q = np.empty_like(P); nfb = 0
    for ri in range(R):
        for f in FOLDS:
            te = FOLD_ROW[ri] == f; ca = ~te; res, pc = e[ri][ca], P[ri][ca]; qg = kth_smallest(res, LEVEL)
            edges = np.unique(np.quantile(pc, np.linspace(0, 1, NBINS + 1)[1:-1])); b_ca = np.searchsorted(edges, pc, side="right"); qb = []
            for b in range(len(edges) + 1):
                rb = res[b_ca == b]
                if rb.size < 20: qb.append(qg); nfb += 1
                else: qb.append(kth_smallest(rb, LEVEL))
            Q[ri][te] = np.array(qb)[np.searchsorted(edges, P[ri][te], side="right")]
    return Q, nfb
def ema_all(P, a):
    out = np.empty_like(P)
    for s, c in zip(STARTS, COUNTS):
        seg = P[:, s:s + c]; out[:, s:s + c], _ = lfilter([a], [1.0, -(1.0 - a)], seg, axis=1, zi=(1.0 - a) * seg[:, :1])
    return out
def first_cross(S, thr):
    res = np.full((R, NC), np.nan)
    for c, (s, n) in enumerate(zip(STARTS, COUNTS)):
        hit = S[:, s:s + n] <= thr; has = hit.any(axis=1); res[has, c] = CYC[s:s + n][hit.argmax(axis=1)[has]]
    return res
TRUE85 = np.array([CYC[s:s + n][Y[s:s + n] <= 85.0][0] if (Y[s:s + n] <= 85.0).any() else np.nan for s, n in zip(STARTS, COUNTS)])
TRUTH = {85.0: TRUE85, 80.0: LIFE}          # 80%: the cycle_life field (end of life); 85%: first scored cycle with true SOH <= 85
print("cells with no true crossing of 85% in scored rows (excluded from the 85% alert):", int(np.isnan(TRUE85).sum()))

SERIES = {}          # (w, m, variant) -> predicted series (R, N)
for (w, m) in AL_COMBOS:
    P = PRED[(w, m)]; SERIES[(w, m, "raw")] = P
    SM = {a: ema_all(P, a) for a in ALPHAS}; CMAE = {a: seg_mean(np.abs(SM[a] - Y)) for a in ALPHAS}; SEL = np.empty_like(P); chosen = []
    for ri in range(R):
        for f in FOLDS:
            other = FOLD_CELL[ri] != f; a_best = ALPHAS[int(np.argmin([CMAE[a][ri, other].mean() for a in ALPHAS]))]; chosen.append(a_best)
            te = FOLD_ROW[ri] == f; SEL[ri, te] = SM[a_best][ri, te]
    SERIES[(w, m, "ema")] = SEL; print(f"EMA alpha choices {w} {m} (on other folds only):", {a: chosen.count(a) for a in ALPHAS})

cov_rows, covg_rows, covc_rows, QB, EV = [], [], [], {}, {}
for (w, m, v), P in SERIES.items():
    Q, nfb = onesided_q(P); QB[(w, m, v)] = Q; L = P - Q
    if nfb: print(f"  {w} {m} {v}: {nfb} bins with <20 calibration rows used the global quantile")
    cov = (Y[None, :] >= L).astype(float); cell_cov = seg_mean(cov).mean(0)
    lo, hi = ci95(wmean(W_ALL, cell_cov)); add_int("2d coverage CIs")
    cov_rows.append(dict(window=w, model=m, variant=v, nominal=LEVEL, n_cells=NC, macro_cov=cell_cov.mean(), ci95_lo=lo, ci95_hi=hi, pooled_cov=cov.mean(), min_cell_cov=cell_cov.min(),
                         share_cells_below_0p80=(cell_cov < 0.80).mean(), mean_q_pp=float(Q.mean())))
    for gname, grp, labs in (("life_tertile", TERT, TERT_LAB), ("stitched", IS_ST.astype(int), ["other", "stitched"])):
        for k, lab in enumerate(labs):
            mk = grp == k; l2, h2 = ci95(wmean(W_for(int(mk.sum())), cell_cov[mk])); add_int("2d coverage CIs")
            covg_rows.append(dict(window=w, model=m, variant=v, group_type=gname, group=lab, n_cells=int(mk.sum()), macro_cov=cell_cov[mk].mean(), ci95_lo=l2, ci95_hi=h2))
    for c in range(NC):
        covc_rows.append(dict(window=w, model=m, variant=v, cell_id=cells[c], life=LIFE[c], stitched=bool(IS_ST[c]), onesided_cov=cell_cov[c]))
    for thr, truth in TRUTH.items():
        for rule, S in (("A_estimate", P), ("B_lower_bound", L)):
            est = first_cross(S, thr); ci, ri_ = np.meshgrid(np.arange(NC), np.arange(R))
            ev = pd.DataFrame(dict(repeat=ri_.ravel(), cell=ci.ravel(), est=est.ravel(), truth=truth[ci.ravel()])); ev = ev[~ev.truth.isna()].copy()
            ev["delay"] = ev.est - ev.truth; ev["pct"] = 100.0 * ev.delay / LIFE[ev.cell.values]; ev["missed"] = ev.est.isna(); ev["early"] = ev.pct < EARLY_PCT
            EV[(w, m, v, thr, rule)] = ev
COV_DF, COVG_DF, COVC_DF = pd.DataFrame(cov_rows), pd.DataFrame(covg_rows), pd.DataFrame(covc_rows)
save_csv(COV_DF, "onesided_coverage.csv"); save_csv(COVG_DF, "onesided_coverage_by_group.csv"); save_csv(COVC_DF, "onesided_coverage_per_cell.csv")
print("\nOne-sided 90% lower-bound coverage (share of rows with y >= L; cell-balanced; approximate, calibration folds' models saw the test fold's cells):")
print(COV_DF.to_string(index=False)); print(COVG_DF.to_string(index=False))

def ev_summary(ev, mask):
    e = ev[mask[ev.cell.values]]; det = e[~e.missed]
    return dict(n_events=len(e), n_cells=int(e.cell.nunique()), n_detected=len(det), missed_rate=e.missed.mean() if len(e) else np.nan,
                early_alarm_rate=e.early.mean() if len(e) else np.nan, median_delay_cycles=det.delay.median(), mean_abs_delay_cycles=det.delay.abs().mean(),
                median_delay_pct_life=det.pct.median(), mean_abs_delay_pct_life=det.pct.abs().mean(), mean_delay_cycles=det.delay.mean(), mean_delay_pct_life=det.pct.mean())
al_rows = []
for (w, m, v, thr, rule), ev in EV.items():
    for sname, mask in (("non_stitched", ~IS_ST), ("stitched", IS_ST)):
        al_rows.append(dict(window=w, model=m, variant=v, threshold=int(thr), rule=rule, subset=sname, **ev_summary(ev, mask)))
ALERT_DF = pd.DataFrame(al_rows); save_csv(ALERT_DF, "alert_rules.csv")
print("\nAlert rules (delay = estimated - true crossing, negative = early; early alarm = more than 5% of life early).")
print("RIGHT-CENSORING: scoring stops at end of life, so a detection that would occur after the last scored row is counted as MISSED, not as a late alert; delay statistics use detected events only.")
print("Thresholds: 85% truth = first scored cycle with true SOH <= 85; 80% truth = cycle_life. Lower bound L = estimate - q (one-sided 90% conformal on signed error). Nothing is chosen on test cells.")
print(ALERT_DF[ALERT_DF.variant == "raw"].to_string(index=False))

cells with no true crossing of 85% in scored rows (excluded from the 85% alert): 0
EMA alpha choices W300 M0 (on other folds only): {0.2: 6, 0.3: 12, 0.5: 0}
EMA alpha choices W300 B4r (on other folds only): {0.2: 18, 0.3: 0, 0.5: 0}
EMA alpha choices W240 M0 (on other folds only): {0.2: 0, 0.3: 10, 0.5: 8}
EMA alpha choices W240 B4r (on other folds only): {0.2: 0, 0.3: 18, 0.5: 0}

One-sided 90% lower-bound coverage (share of rows with y >= L; cell-balanced; approximate, calibration folds' models saw the test fold's cells):
window model variant  nominal  n_cells  macro_cov  ci95_lo  ci95_hi  pooled_cov  min_cell_cov  share_cells_below_0p80  mean_q_pp
  W300    M0     raw   0.9000       84     0.9459   0.9061   0.9781      0.8709        0.0862                  0.0714     2.8392
  W300    M0     ema   0.9000       84     0.9470   0.9073   0.9790      0.8716        0.0811                  0.0714     3.0261
  W300   B4r     raw   0.9000       84     0.9361   0.9005   0.9658      0.8764   

## 3. Training ablations of M0
Fixed config from D69 (E=120, batch 256, 5 seeds 42-46, stride-5 cell-balanced rows, abs head), Protocol D, same 18 outer sets and same scoring as 05b. Each variant changes one thing relative to `ref`. No tuning is run. All comparisons are descriptive; the number of intervals is printed.

In [9]:
from sklearn.ensemble import RandomForestRegressor
NJ = max(1, min(10, os.cpu_count() or 1)); E_EP = COMMON["E"]
CUR = {}; MEMBER_AUDIT = dict(members=0, trees=0)
def ensure(cond, msg, detail=""):
    if not cond: chk("FAIL", msg, detail)

_DQKEY = hashlib.sha1(ROWID.tobytes()).hexdigest()[:10]
def get_dq(win):
    i_close = int(sc.grid_index(GRID_V[win])); width = i_close + 1; fd = CACHE / f"dq_{win}_{_DQKEY}.npy"
    if fd.exists(): dq = np.load(fd)
    else:
        dq = np.zeros((N, width), np.float32)
        for c in cells:
            pos = CELL_POS[c]; blk = np.asarray(qdlin[ROWID[pos]], dtype=np.float32)
            out = np.asarray(sc.build_delta_q(blk, qref[QREF_ROW[c]], 0, i_close))
            ensure(out.shape == (len(pos), width), f"build_delta_q shape {win} {c}", out.shape); dq[pos] = out
        save_npy(fd, dq)
    for r in np.random.default_rng(SEED_BOOT).choice(N, 3, replace=False):
        one = np.asarray(sc.build_delta_q(np.asarray(qdlin[ROWID[r]], dtype=np.float32), qref[QREF_ROW[CELLID[r]]], 0, i_close))
        need(np.abs(one.ravel() - dq[r]).max() < 1e-6 and abs(float(dq[r, 0])) < 1e-6 and dq.shape[1] == width, f"delta-Q check {win} row {r}")
    return dq
def use_window(win):
    CUR.clear(); gc.collect(); torch.cuda.empty_cache()
    dq = get_dq(win); width = dq.shape[1]; g = np.asarray(sd.sub_idx(width)).astype(int)
    need(len(g) == 32, "sub_idx gives 32 grid points", len(g))
    CUR.update(win=win, dq=dq, width=width, grid=g, Xt=torch.as_tensor(dq, device=DEVICE, dtype=torch.float32),
               q0t=torch.as_tensor(Q0, device=DEVICE, dtype=torch.float32), q0c=torch.full((N,), float(Q0.mean()), device=DEVICE, dtype=torch.float32),
               yt=torch.as_tensor(Y, device=DEVICE, dtype=torch.float32))
    ram_report(f"({win})")

def make_cfg(hp, E, huber_delta=1.0):
    v = dict(WD=hp["wd"], BATCH=256, CLIP=1.0, HUBER_DELTA=float(huber_delta), NOISE_SIGMA=0.02, NOMINAL_AH=1.1, WARMUP_FRAC=0.05)
    miss = [k for k in CFG_KEYS if k not in v]
    if miss: raise RuntimeError(f"cfg key(s) {miss} used by soh_deep_v2 but not set in make_cfg")
    return v

def train_rows(cs, seed, stride=5, weights="cell"):
    # stride 5: every 5th scored row per cell with a random per-cell offset (exactly as 05b); stride 1: every scored row
    rng = np.random.default_rng(seed); pos, w = [], []
    for c in sorted(cs):
        p = CELL_POS[c]
        if stride == 5: q = p[int(rng.integers(0, 5))::5]
        elif stride == 1: q = p
        else: raise ValueError(stride)
        pos.append(q); w.append(np.full(len(q), 1.0 / len(q)) if weights == "cell" else np.ones(len(q)))
    pos, w = np.concatenate(pos).astype(np.int64), np.concatenate(w).astype(np.float32)
    ensure(len(pos) > 0 and set(CELLID[pos]) <= set(cs), "train rows valid and inside training cells")
    return pos, w

def split_preds(out, lens):
    arrs = [np.asarray(a).reshape(-1) for a in out]
    ensure(len(arrs) == len(lens) and all(len(a) == L for a, L in zip(arrs, lens)), "prediction lengths match eval rows")
    return [a.astype(np.float64) for a in arrs]
_PARAM_CHECKED = []
def deep_chunk(win, hp, chunk, var):
    for m in chunk: ensure(not set(m["train"]) & set(m["eval"]), "training members never include their own evaluation cells", f"{m['tag']}")
    tr = [train_rows(m["train"], m["seed"], var["stride"], var["weights"]) for m in chunk]; ev = [eval_pos(m["eval"]) for m in chunk]
    cfg = make_cfg(hp, E_EP, var["huber_delta"])
    mem = [dict(seed=m["seed"], train_rows=r, train_w=w, val_rows=e, val_codes=pd.factorize(CELLID[e])[0].astype(np.int64)) for m, (r, w), e in zip(chunk, tr, ev)]
    q0t = CUR["q0c"] if var["noq0"] else CUR["q0t"]
    ret = sd.train_members(X=CUR["Xt"], q0=q0t, y=CUR["yt"], members=mem, kind="M0", head="abs", lr=hp["lr"], epochs=E_EP, cfg=cfg, device=DEVICE, dtype=torch.float32, log_every=10 ** 6)
    ensure(isinstance(ret, dict) and "val_pred" in ret, "train_members returned val_pred")
    preds = split_preds(ret["val_pred"], [len(e) for e in ev])
    if ret.get("nan_members"): chk("WARN", "NaN members", f"{win}/{var['name']}: members {list(ret['nan_members'])[:10]}")
    if not _PARAM_CHECKED:
        _PARAM_CHECKED.append(1); npar = int(sd.count_params(sd.make_single(kind="M0", L=CUR["width"], seed=0))); need(npar == 2969, "count_params M0", f"{npar} vs 2969", level="WARN")
    MEMBER_AUDIT["members"] += len(chunk); del ret, mem; gc.collect(); torch.cuda.empty_cache()
    return preds
def run_chunked(fn, items, G, label):
    out, i = [], 0
    while i < len(items):
        ch = items[i:i + G]
        try: out += fn(ch); i += len(ch)
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            if G == 1: raise
            G = max(1, G // 2); chk("WARN", "CUDA OOM", f"{label}: retrying with G={G}")
    return out
def gcap(win): return 72 if win == "W300" else 36

def pred_frame(win, name, rep, of, seed, ep, pred):
    return pd.DataFrame(dict(window=win, variant=name, repeat=rep, outer_fold=of, seed=seed, row_id=ROWID[ep], cell_id=CELLID[ep], pred_pp=pred))
def ckey(kind, win, spec, hp, seeds, sets):
    d = dict(module=MODULE_SHA, nb=NB_VERSION, kind=kind, win=win, spec=spec, hp=hp, seeds=list(seeds), E=E_EP,
             sets=[(s["repeat"], s["outer_fold"], hashlib.sha1("|".join(s["train"] + ["#"] + s["test"]).encode()).hexdigest()[:10]) for s in sets], trees=COMMON["trees"])
    return hashlib.sha1(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:20]

def run_deep(win, var, sets, tag):
    hp = FROZEN_CFG[win]; fp = CACHE / f"{tag}_deep_{win}_{var['name']}_{ckey('deep', win, var, hp, SEEDS, sets)}.parquet"
    if fp.exists():
        print(f"  cached: {fp.name}"); return pd.read_parquet(fp)
    t0 = time.time()
    mem = [dict(train=s["train"], eval=s["test"], seed=sd_, tag=(s["repeat"], s["outer_fold"])) for s in sets for sd_ in SEEDS]
    preds = run_chunked(lambda ch: deep_chunk(win, hp, ch, var), mem, gcap(win), f"{win}/{var['name']}")
    fr = pd.concat([pred_frame(win, var["name"], m["tag"][0], m["tag"][1], m["seed"], eval_pos(m["eval"]), p) for m, p in zip(mem, preds)], ignore_index=True)
    need(bool(np.isfinite(fr.pred_pp.values).all()), f"no NaN/inf predictions: {win}/{var['name']}")
    save_parquet(fr, fp); print(f"  trained {win}/{var['name']}: {len(mem)} members in {time.time() - t0:.0f}s"); return fr

TREE_X = {}
def tree_X(win, logcycle):
    k = (win, logcycle)
    if k not in TREE_X:
        X = np.column_stack([CUR["dq"][:, CUR["grid"]], Q0]); TREE_X[k] = np.column_stack([X, np.log(CYC)]) if logcycle else X
    return TREE_X[k]
def tree_fit_predict(win, leaf, train_cs, eval_cs, seed, logcycle):
    ensure(not set(train_cs) & set(eval_cs), "tree training cells exclude their evaluation cells")
    pos, w = train_rows(train_cs, seed, 5, "cell"); X = tree_X(win, logcycle); off = 100.0 * Q0 / 1.1
    est = RandomForestRegressor(n_estimators=COMMON["trees"], min_samples_leaf=leaf, max_features=0.33, n_jobs=NJ, random_state=seed)
    est.fit(X[pos], (Y - off)[pos], sample_weight=w); ep = eval_pos(eval_cs); MEMBER_AUDIT["trees"] += 1
    return ep, est.predict(X[ep]) + off[ep]
def run_tree(win, name, logcycle, sets, tag):
    leaf = FROZEN_CFG[win]["b4r_leaf"]; fp = CACHE / f"{tag}_tree_{win}_{name}_{ckey('tree', win, dict(name=name, leaf=leaf, log=logcycle), dict(leaf=leaf), [COMMON['tree_seed']], sets)}.parquet"
    if fp.exists():
        print(f"  cached: {fp.name}"); return pd.read_parquet(fp)
    t0 = time.time(); out = []
    for s in sets:
        ep, p = tree_fit_predict(win, leaf, s["train"], s["test"], COMMON["tree_seed"], logcycle); out.append(pred_frame(win, name, s["repeat"], s["outer_fold"], COMMON["tree_seed"], ep, p))
    fr = pd.concat(out, ignore_index=True); need(bool(np.isfinite(fr.pred_pp.values).all()), f"no NaN/inf predictions: {win}/{name}")
    save_parquet(fr, fp); print(f"  fitted {win}/{name}: {len(sets)} forests in {time.time() - t0:.0f}s"); return fr

def ens_array(fr, nseeds, label):
    pos = ROW_INDEX.get_indexer(fr.row_id.values); ensure(bool((pos >= 0).all()), f"row_ids are development rows ({label})")
    arr = np.full((R, N), np.nan); rv = fr["repeat"].values
    for ri, r in enumerate(REPS):
        m = rv == r; p = pos[m]; s_ = np.bincount(p, weights=fr.pred_pp.values[m], minlength=N); n_ = np.bincount(p, minlength=N)
        ensure(bool(np.all(n_ == nseeds)), f"every row predicted exactly once per seed in repeat {r} ({label})", f"counts {np.unique(n_)}")
        arr[ri] = s_ / n_
    ensure(arr.shape == (R, N) and not np.isnan(arr).any(), f"ensemble array complete ({label})")
    # each test fold only predicts its own test cells
    for (r, f), g in fr.groupby(["repeat", "outer_fold"]):
        te = set(pD[(pD.repeat == r) & (pD.outer_fold == f) & (pD.role == "test")].cell_id); ensure(set(g.cell_id) == te, f"predictions cover exactly the test cells of ({r},{f}) ({label})")
    return arr
def verify_noq0():
    rows = train_rows(SETS[0]["train"], 42)[0]
    s_ = sd.fit_scalers(CUR["Xt"], CUR["q0c"], CUR["yt"], [rows], "abs", 1.1, DEVICE, torch.float32)
    d, q = sd.make_inputs(CUR["Xt"], CUR["q0c"], torch.as_tensor(rows[:512], device=DEVICE)[None, :], s_)
    need(bool((q == 0).all()) and float(s_["qs"][0]) == 1.0 and bool(torch.isfinite(d).all()) and bool(torch.isfinite(q).all()),
         "noQ0: scaled Q0 input is exactly 0, scaler std guard gives 1.0, no NaN/inf", f"max|q|={float(q.abs().max()):.1e}, qs={float(s_['qs'][0])}")
chk("PASS", "training adapters defined", f"seeds {SEEDS}, E={E_EP}, trees {COMMON['trees']}, tree jobs {NJ}")

[PASS] training adapters defined | seeds [42, 43, 44, 45, 46], E=120, trees 300, tree jobs 10


In [10]:
VARIANTS = [dict(name="ref", noq0=False, weights="cell", huber_delta=1.0, stride=5),
            dict(name="noQ0", noq0=True, weights="cell", huber_delta=1.0, stride=5),
            dict(name="sample_weighted", noq0=False, weights="uniform", huber_delta=1.0, stride=5),
            dict(name="mse", noq0=False, weights="cell", huber_delta=1e6, stride=5),
            dict(name="stride1", noq0=False, weights="cell", huber_delta=1.0, stride=1)]
SKIPPED = [dict(variant="size_sweep", status="SKIPPED", reason=("make_single/make_batched expose no width or channel argument; M0 is fixed at 33-44-32-1 (2,969 parameters) in the frozen module" if not SIZE_EXPOSED
                                                                  else "a size-like parameter was found but a size sweep was not specified for this phase; not run")),
           dict(variant="temperature", status="SKIPPED", reason="temperature never extracted; excluded from the primary model by design"),
           dict(variant="internal_resistance", status="SKIPPED", reason="IR never extracted; excluded from the primary model by design"),
           dict(variant="policy_features", status="SKIPPED", reason="policy never extracted as a model input; excluded from the primary model by design")]
print("SKIPPED variants:"); [print("  ", s["variant"], "-", s["reason"]) for s in SKIPPED]
print("mse variant: HUBER_DELTA = 1e6 pp. F.huber_loss equals 0.5*x^2 whenever |x| <= delta, which is also what delta=1 gives inside its quadratic zone, so the gradient scale (and gradient clipping) match the reference and only the linear tails of the Huber loss are removed.")
ABL, ABL_FRAMES = {}, {}
for w in WINS:
    print(f"\n=== {w} ==="); use_window(w); verify_noq0()
    for var in VARIANTS:
        fr = run_deep(w, var, SETS, "abl"); ABL_FRAMES[(w, f"M0:{var['name']}")] = fr; ABL[(w, f"M0:{var['name']}")] = ens_array(fr, len(SEEDS), f"{w} M0:{var['name']}")
    for name, lg in (("ref", False), ("+log_cycle", True)):
        fr = run_tree(w, name, lg, SETS, "abl"); ABL_FRAMES[(w, f"B4r:{name}")] = fr; ABL[(w, f"B4r:{name}")] = ens_array(fr, 1, f"{w} B4r:{name}")
    ABL[(w, "M0:5b_nested")] = PRED[(w, "M0")]; ABL[(w, "B4r:5b_saved")] = PRED[(w, "B4r")]
    gc.collect(); torch.cuda.empty_cache()

def variant_metrics(P):
    ae = np.abs(P - Y); cell = seg_mean(ae).mean(0); tail = (CYC >= 0.8 * LIFE_ROW)
    num = np.add.reduceat(ae * tail, STARTS, axis=1); den = np.add.reduceat(tail.astype(float), STARTS)
    with np.errstate(invalid="ignore", divide="ignore"): last20 = np.nanmean(num / den)
    return dict(cell=cell, macro=float(cell.mean()), macro_excl_stitched=float(cell[~IS_ST].mean()), last20_macro=float(last20), bias=float((P - Y).mean()))
BASE_OF = {"M0:noQ0": "M0:ref", "M0:sample_weighted": "M0:ref", "M0:mse": "M0:ref", "M0:stride1": "M0:ref", "M0:ref": "M0:5b_nested", "B4r:+log_cycle": "B4r:ref", "B4r:ref": "B4r:5b_saved"}
ORDER = ["M0:5b_nested", "M0:ref", "M0:noQ0", "M0:sample_weighted", "M0:mse", "M0:stride1", "B4r:5b_saved", "B4r:ref", "B4r:+log_cycle"]
abl_rows, FOREST = [], []
for w in WINS:
    VM = {v: variant_metrics(ABL[(w, v)]) for v in ORDER}
    for v in ORDER:
        row = dict(window=w, variant=v, base=BASE_OF.get(v, ""), macro_mae=VM[v]["macro"], macro_excl_stitched=VM[v]["macro_excl_stitched"], last20_macro_mae=VM[v]["last20_macro"], pooled_bias=VM[v]["bias"])
        if v in BASE_OF:
            d = VM[v]["cell"] - VM[BASE_OF[v]]["cell"]; lo, hi = ci95(wmean(W_ALL, d)); add_int("3 paired ablation intervals")
            row.update(diff_vs_base=float(d.mean()), ci95_lo=float(lo), ci95_hi=float(hi), ci_excludes_zero=bool(lo > 0 or hi < 0)); FOREST.append(dict(window=w, variant=v, base=BASE_OF[v], diff=float(d.mean()), lo=float(lo), hi=float(hi)))
        abl_rows.append(row)
ABL_DF = pd.DataFrame(abl_rows); save_csv(ABL_DF, "ablation_results.csv"); save_csv(pd.DataFrame(SKIPPED), "ablation_skipped.csv"); FOREST_DF = pd.DataFrame(FOREST)
print(f"\nDESCRIPTIVE ONLY. {INTERVALS.get('3 paired ablation intervals', 0)} paired cluster-bootstrap intervals (84 cells, 5000 draws, seed 42) are shown; some will exclude zero by chance.")
print("M0:ref vs M0:5b_nested = fixed D69 config against the nested-tuned M0 of 05b. B4r:ref = fixed leaf from the D69-style pooled rule, no nested tuning, same 32-point grid + Q0, 300 trees, max_features 0.33.")
print("B4r:+log_cycle adds log(cycle_number) to B4r (the frozen module has no extra-scalar input, so this ablation is run on the tree model, not on M0).")
print(ABL_DF.to_string(index=False))

SKIPPED variants:
   size_sweep - make_single/make_batched expose no width or channel argument; M0 is fixed at 33-44-32-1 (2,969 parameters) in the frozen module
   temperature - temperature never extracted; excluded from the primary model by design
   internal_resistance - IR never extracted; excluded from the primary model by design
   policy_features - policy never extracted as a model input; excluded from the primary model by design
mse variant: HUBER_DELTA = 1e6 pp. F.huber_loss equals 0.5*x^2 whenever |x| <= delta, which is also what delta=1 gives inside its quadratic zone, so the gradient scale (and gradient clipping) match the reference and only the linear tails of the Huber loss are removed.

=== W300 ===
[PASS] delta-Q check W300 row 36631
[PASS] delta-Q check W300 row 4994
[PASS] delta-Q check W300 row 43312
[PASS] sub_idx gives 32 grid points | 32
RAM available 3.3 GB (W300); top 8: python.exe 1.5GB, Code.exe 1.0GB, chrome.exe 0.5GB, Code.exe 0.5GB, python.exe 0.5GB, explor

## 4. Extreme-life holdout
Per window, the shortest and longest ~10% of the 84 development cells (ties at the cut included) are held out, together with a policy-twin purge: every other cell sharing a `policy_norm` string with a held-out cell is removed from training. M0 (fixed D69 config, 5 seeds) and B4r (leaf by the D69-style pooled rule) are trained on the remaining cells and score the held-out cells. The short and long groups are run as separate holdouts (`EXTREME_MODE = "separate"`); set `"joint"` to hold both out at once.

In [11]:
EXTREME_MODE = "separate"
K_EXT = int(round(0.10 * NC)); _ls = np.sort(LIFE)
short_cut, long_cut = _ls[K_EXT - 1], _ls[-K_EXT]
SHORT = [c for c in cells if LIFE[CIDX[c]] <= short_cut]; LONG = [c for c in cells if LIFE[CIDX[c]] >= long_cut]
print(f"held-out groups: shortest {len(SHORT)} cells (life <= {short_cut:.0f}), longest {len(LONG)} cells (life >= {long_cut:.0f})")
need(8 <= len(SHORT) <= 9 and 8 <= len(LONG) <= 9, "8 or 9 cells in each extreme group", (len(SHORT), len(LONG)))
GROUPS = {"short": SHORT, "long": LONG} if EXTREME_MODE == "separate" else {"joint": SHORT + LONG}
EXT_SETS, EXT_INFO = [], []
for gi, (gname, held) in enumerate(GROUPS.items()):
    hp_ = {POLICY[c] for c in held}; twins = sorted(c for c in cells if c not in held and POLICY[c] in hp_)
    train = [c for c in cells if c not in held and c not in twins]
    need(not set(train) & set(held) and not set(train) & set(twins), f"{gname}: no held-out cell or policy twin in training")
    need(not any(POLICY[c] in hp_ for c in train), f"{gname}: no training cell shares a policy_norm with a held-out cell")
    need(not (set(train) | set(held)) & SENS_CELLS and not any(c.startswith("b3") for c in train + held), f"{gname}: no batch-3 / sensitivity cells")
    tl = LIFE[[CIDX[c] for c in train]]; EXT_SETS.append(dict(repeat=0, outer_fold=gi, train=train, test=sorted(held), name=gname))
    EXT_INFO.append(dict(group=gname, n_held=len(held), n_policies_held=len(hp_), n_twins_removed=len(twins), n_train=len(train), train_life_min=float(tl.min()), train_life_max=float(tl.max()), twins_removed=" ".join(twins)))
    print(f"  {gname}: held {len(held)} cells ({len(hp_)} policies), removed {len(twins)} policy twins, trained on {len(train)} cells (life {tl.min():.0f}-{tl.max():.0f})")
EXT_INFO_DF = pd.DataFrame(EXT_INFO); save_csv(EXT_INFO_DF, "extreme_holdout_counts.csv")

def ens_vec(fr, nseeds, label):
    pos = ROW_INDEX.get_indexer(fr.row_id.values); ensure(bool((pos >= 0).all()), f"row_ids are development rows ({label})"); vec = np.full(N, np.nan)
    s_ = np.bincount(pos, weights=fr.pred_pp.values, minlength=N); n_ = np.bincount(pos, minlength=N); m = n_ > 0
    ensure(bool(np.all(n_[m] == nseeds)), f"every held-out row predicted exactly once per seed ({label})"); vec[m] = s_[m] / n_[m]; return vec
ext_cell_rows = []
VCELL = {(w, v): variant_metrics(ABL[(w, v)])["cell"] for w in WINS for v in ("M0:ref", "B4r:ref")}
for w in WINS:
    print(f"\n=== extreme holdout {w} ==="); use_window(w)
    fm = run_deep(w, VARIANTS[0], EXT_SETS, "ext"); ft = run_tree(w, "ext_B4r", False, EXT_SETS, "ext")
    for s in EXT_SETS:
        gm, gt = fm[fm.outer_fold == s["outer_fold"]], ft[ft.outer_fold == s["outer_fold"]]
        need(set(gm.cell_id) == set(s["test"]) and set(gt.cell_id) == set(s["test"]), f"{w}/{s['name']}: predictions only for the held-out cells")
        PM, PT = ens_vec(gm, len(SEEDS), f"{w} {s['name']} M0"), ens_vec(gt, 1, f"{w} {s['name']} B4r"); tl = (min(LIFE[[CIDX[c] for c in s["train"]]]), max(LIFE[[CIDX[c] for c in s["train"]]]))
        for c in s["test"]:
            ci_ = CIDX[c]; sl = slice(STARTS[ci_], STARTS[ci_] + COUNTS[ci_]); L_ = LIFE[ci_]
            dist = max(0.0, tl[0] - L_, L_ - tl[1])
            ext_cell_rows.append(dict(window=w, group=s["name"], cell_id=c, batch=int(BATCH[ci_]), stitched=bool(IS_ST[ci_]), life=L_, policy_norm=POLICY[c], train_life_min=tl[0], train_life_max=tl[1],
                                      dist_from_train_range=dist, side="below" if L_ < tl[0] else "above" if L_ > tl[1] else "inside",
                                      M0_extreme_mae=float(np.abs(PM[sl] - Y[sl]).mean()), M0_extreme_bias=float((PM[sl] - Y[sl]).mean()),
                                      M0_D5b_mae=float(CMX[(w, "M0")]["mae"][ci_]), M0_Dref_mae=float(VCELL[(w, "M0:ref")][ci_]),
                                      B4r_extreme_mae=float(np.abs(PT[sl] - Y[sl]).mean()), B4r_extreme_bias=float((PT[sl] - Y[sl]).mean()),
                                      B4r_D5b_mae=float(CMX[(w, "B4r")]["mae"][ci_]), B4r_Dref_mae=float(VCELL[(w, "B4r:ref")][ci_])))
EXT_CELLS = pd.DataFrame(ext_cell_rows); save_csv(EXT_CELLS, "extreme_holdout_cells.csv")
g_ = EXT_CELLS.groupby(["window", "group"]); EXT_SUM = g_.agg(n_held=("cell_id", "size"), mean_life=("life", "mean"), mean_dist_from_train=("dist_from_train_range", "mean"), max_dist_from_train=("dist_from_train_range", "max"),
    M0_extreme=("M0_extreme_mae", "mean"), M0_D5b=("M0_D5b_mae", "mean"), M0_Dref=("M0_Dref_mae", "mean"), B4r_extreme=("B4r_extreme_mae", "mean"), B4r_D5b=("B4r_D5b_mae", "mean"), B4r_Dref=("B4r_Dref_mae", "mean")).reset_index()
EXT_SUM["M0_ratio_vs_D5b"] = EXT_SUM.M0_extreme / EXT_SUM.M0_D5b; EXT_SUM["B4r_ratio_vs_D5b"] = EXT_SUM.B4r_extreme / EXT_SUM.B4r_D5b
EXT_SUM = EXT_SUM.merge(EXT_INFO_DF[["group", "n_twins_removed", "n_train"]], on="group"); save_csv(EXT_SUM, "extreme_holdout_summary.csv")
print("\nCell-macro MAE of the held-out groups (extreme = trained without them and their policy twins; D5b = the same cells' Protocol D out-of-fold MAE from 05b; Dref = fixed-config re-run):")
print(EXT_SUM.to_string(index=False)); print("\nPer cell:"); print(EXT_CELLS.drop(columns=["policy_norm"]).to_string(index=False))

held-out groups: shortest 8 cells (life <= 457), longest 8 cells (life >= 1051)
[PASS] 8 or 9 cells in each extreme group | (8, 8)
[PASS] short: no held-out cell or policy twin in training
[PASS] short: no training cell shares a policy_norm with a held-out cell
[PASS] short: no batch-3 / sensitivity cells
  short: held 8 cells (8 policies), removed 0 policy twins, trained on 76 cells (life 458-2237)
[PASS] long: no held-out cell or policy twin in training
[PASS] long: no training cell shares a policy_norm with a held-out cell
[PASS] long: no batch-3 / sensitivity cells
  long: held 8 cells (5 policies), removed 1 policy twins, trained on 75 cells (life 148-1017)

=== extreme holdout W300 ===
[PASS] delta-Q check W300 row 36631
[PASS] delta-Q check W300 row 4994
[PASS] delta-Q check W300 row 43312
[PASS] sub_idx gives 32 grid points | 32
RAM available 3.6 GB (W300); top 8: python.exe 2.8GB, Code.exe 0.8GB, chrome.exe 0.5GB, chrome.exe 0.4GB, explorer.exe 0.4GB, MemCompression 0.4GB, Cod

## 5. Figures, report, final checks

In [12]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 10})
COL = {"M0": "#0072B2", "B4r": "#009E73", "M1p": "#D55E00"}
def save_fig(fig, name):
    p = _guard(FIG / name); fig.savefig(p, dpi=150, bbox_inches="tight", facecolor="white"); WRITTEN.append(p); plt.show(); plt.close(fig); return p

# (a) per-cell bias vs life, M0 and B4r, W300, stitched highlighted
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, m in zip(axs, ("M0", "B4r")):
    cm = CMX[("W300", m)]
    for b, mk in ((1, "o"), (2, "s")):
        k = (BATCH == b) & ~IS_ST; ax.scatter(LIFE[k], cm["bias"][k], marker=mk, s=22, color=COL[m], alpha=0.75, label=f"batch {b}")
    ax.scatter(LIFE[IS_ST], cm["bias"][IS_ST], s=60, facecolor="none", edgecolor="#D55E00", lw=1.6, label="stitched (b1c0-4)")
    sp = SPEAR[(SPEAR.window == "W300") & (SPEAR.model == m)].set_index("subset")
    ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("cell life (cycles)")
    ax.set_title(f"{m} W300: rho all {sp.loc['all84', 'spearman_bias_vs_life']:+.2f} [{sp.loc['all84', 'ci95_lo']:+.2f}, {sp.loc['all84', 'ci95_hi']:+.2f}], excl. stitched {sp.loc['excl_stitched', 'spearman_bias_vs_life']:+.2f}", fontsize=8)
axs[0].set_ylabel("mean signed error (pp)  [+ = over-estimated]"); axs[0].legend(frameon=False, fontsize=7)
save_fig(fig, "fig_a_bias_vs_life_W300.png")

# (b) trajectories around the junction, M0 W300, 5 stitched cells
fig, axs = plt.subplots(2, 3, figsize=(12, 6.4), constrained_layout=True); Pm = PRED[("W300", "M0")]
for ax, cid in zip(axs.ravel(), STITCHED):
    ci_ = CIDX[cid]; s, n = STARTS[ci_], COUNTS[ci_]; cyc = CYC[s:s + n]; ju = JUNC[cid]["used"]; k = (cyc >= ju - 300) & (cyc <= ju + 300)
    for ri in range(R): ax.plot(cyc[k], Pm[ri, s:s + n][k], color=COL["M0"], alpha=0.25, lw=0.8)
    ax.plot(cyc[k], Pm[:, s:s + n].mean(0)[k], color=COL["M0"], lw=1.3, label="M0 W300 (mean of repeats)"); ax.plot(cyc[k], Y[s:s + n][k], color="k", lw=1.3, label="true")
    ax.axvline(ju, color="#D55E00", ls="--", lw=1.2, label=f"junction {ju} ({JUNC[cid]['rule']})")
    r0 = JUNC_DF[(JUNC_DF.window == "W300") & (JUNC_DF.model == "M0") & (JUNC_DF.cell_id == cid)].iloc[0]
    ax.set_title(f"{cid}: true step {r0.true_step_pp:+.2f}, predicted {r0.pred_step_pp:+.2f} pp", fontsize=8); ax.set_xlabel("cycle"); ax.set_ylabel("SOH (%)"); ax.legend(frameon=False, fontsize=6)
axs.ravel()[-1].axis("off")
save_fig(fig, "fig_b_junction_trajectories_M0_W300.png")

# (c) ablation forest plot
F = FOREST_DF.copy(); F["label"] = [f"{r.window}: {r.variant} - {r.base}" for r in F.itertuples()]; yy = np.arange(len(F))[::-1]
fig, ax = plt.subplots(figsize=(8, 0.34 * len(F) + 1.6))
ax.errorbar(F["diff"], yy, xerr=[F["diff"] - F["lo"], F["hi"] - F["diff"]], fmt="o", capsize=2, color="#0072B2"); ax.axvline(0, color="k", lw=0.8)
ax.set_yticks(yy); ax.set_yticklabels(F["label"], fontsize=7); ax.set_xlabel("paired difference in cell-macro MAE (pp); negative favours the first variant; 95% cluster bootstrap")
ax.set_title(f"Ablations (descriptive; {len(F)} intervals)")
save_fig(fig, "fig_c_ablation_forest.png")

# (d) alert-delay histograms: estimate rule vs lower-bound rule, M0, raw
fig, axs = plt.subplots(2, 2, figsize=(10, 6.2), sharey="row")
allv = np.concatenate([EV[k].pct.dropna().values for k in EV if k[0] in WINS and k[1] == "M0" and k[2] == "raw"]); lo_, hi_ = np.percentile(allv, [1, 99]); bins = np.linspace(min(lo_, -15), max(hi_, 15), 41)
for i, thr in enumerate((85.0, 80.0)):
    for j, w in enumerate(WINS):
        ax = axs[i, j]; a_, b_ = EV[(w, "M0", "raw", thr, "A_estimate")], EV[(w, "M0", "raw", thr, "B_lower_bound")]
        ax.hist(a_.pct.dropna(), bins=bins, color=COL["M0"], alpha=0.55, label=f"estimate <= {int(thr)} (missed {int(a_.missed.sum())}/{len(a_)})")
        ax.hist(b_.pct.dropna(), bins=bins, histtype="step", color="#D55E00", lw=1.5, label=f"lower bound <= {int(thr)} (missed {int(b_.missed.sum())}/{len(b_)})")
        ax.axvline(0, color="k", lw=0.8); ax.axvline(EARLY_PCT, color="grey", ls="--", lw=0.8); ax.set_title(f"M0 {w}, {int(thr)}% threshold"); ax.legend(frameon=False, fontsize=7)
        if i == 1: ax.set_xlabel("delay (% of life; negative = early; detected events only)")
axs[0, 0].set_ylabel("events (cell x repeat)"); axs[1, 0].set_ylabel("events (cell x repeat)")
save_fig(fig, "fig_d_alert_delay_hist.png")

# (e) extreme holdout bar chart
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, w in zip(axs, WINS):
    t = EXT_SUM[EXT_SUM.window == w].reset_index(drop=True); x = np.arange(len(t)); wd = 0.2
    for k, (col, lab, c, hatch) in enumerate((("M0_extreme", "M0 extreme holdout", COL["M0"], None), ("M0_D5b", "M0 Protocol D (5b)", COL["M0"], "//"), ("B4r_extreme", "B4r extreme holdout", COL["B4r"], None), ("B4r_D5b", "B4r Protocol D (5b)", COL["B4r"], "//"))):
        ax.bar(x + (k - 1.5) * wd, t[col], wd, color=c, alpha=0.85 if hatch is None else 0.4, hatch=hatch, label=lab)
    ax.set_xticks(x); ax.set_xticklabels([f"{g} (n={n})" for g, n in zip(t.group, t.n_held)]); ax.set_title(f"{w}: held-out cells, cell-macro MAE"); ax.set_xlabel("held-out group")
axs[0].set_ylabel("MAE (pp)"); axs[0].legend(frameon=False, fontsize=7)
save_fig(fig, "fig_e_extreme_holdout.png")
for p in sorted(FIG.glob("*.png")): need(p.stat().st_size > 0, f"figure written: {p.name}", f"{p.stat().st_size / 1024:.0f} KB")

C:\Users\kulje\AppData\Local\Temp\ipykernel_42488\786492617.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  p = _guard(FIG / name); fig.savefig(p, dpi=150, bbox_inches="tight", facecolor="white"); WRITTEN.append(p); plt.show(); plt.close(fig); return p
C:\Users\kulje\AppData\Local\Temp\ipykernel_42488\786492617.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  p = _guard(FIG / name); fig.savefig(p, dpi=150, bbox_inches="tight", facecolor="white"); WRITTEN.append(p); plt.show(); plt.close(fig); return p


[PASS] figure written: fig_a_bias_vs_life_W300.png | 71 KB
[PASS] figure written: fig_b_junction_trajectories_M0_W300.png | 265 KB
[PASS] figure written: fig_c_ablation_forest.png | 68 KB
[PASS] figure written: fig_d_alert_delay_hist.png | 92 KB
[PASS] figure written: fig_e_extreme_holdout.png | 48 KB


C:\Users\kulje\AppData\Local\Temp\ipykernel_42488\786492617.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  p = _guard(FIG / name); fig.savefig(p, dpi=150, bbox_inches="tight", facecolor="white"); WRITTEN.append(p); plt.show(); plt.close(fig); return p


In [ ]:
# ---- final checks
def _no_nan(df, cols, label):
    bad = int(df[cols].isna().sum().sum()); need(bad == 0, f"no NaN in headline table: {label}", f"{bad} NaN cells")
_no_nan(PER_CELL, ["mae", "mean_signed_err", "within_cell_resid_sd", "signed_err_first_half", "signed_err_second_half"], "per_cell_bias")
_no_nan(SPEAR, ["spearman_bias_vs_life", "ci95_lo", "ci95_hi"], "bias_vs_life_spearman"); _no_nan(TERT_BIAS, ["mean_signed_err", "macro_mae"], "bias_by_life_tertile")
_no_nan(AVG_DF, ["avg_macro_mae", "other_macro_mae", "diff", "ci95_lo", "ci95_hi"], "ensemble_avg_M0_B4r")
_no_nan(COV_DF, ["macro_cov", "ci95_lo", "ci95_hi", "pooled_cov"], "onesided_coverage"); _no_nan(COVG_DF, ["macro_cov", "ci95_lo", "ci95_hi"], "onesided_coverage_by_group")
_no_nan(ALERT_DF, ["missed_rate", "early_alarm_rate"], "alert_rules (rates)")
_no_nan(ABL_DF, ["macro_mae", "macro_excl_stitched", "last20_macro_mae"], "ablation_results"); _no_nan(ABL_DF[ABL_DF.base != ""], ["diff_vs_base", "ci95_lo", "ci95_hi"], "ablation_results (paired)")
_no_nan(EXT_SUM, ["M0_extreme", "M0_D5b", "B4r_extreme", "B4r_D5b"], "extreme_holdout_summary"); _no_nan(EXT_CELLS, ["M0_extreme_mae", "B4r_extreme_mae", "dist_from_train_range"], "extreme_holdout_cells")
_cc = ["macro_cov", "ci95_lo", "ci95_hi"]
need(bool(((COV_DF[_cc + ["pooled_cov"]] >= 0) & (COV_DF[_cc + ["pooled_cov"]] <= 1)).all().all()) and bool(((COVG_DF[_cc] >= 0) & (COVG_DF[_cc] <= 1)).all().all()) and bool(((COVC_DF.onesided_cov >= 0) & (COVC_DF.onesided_cov <= 1)).all()), "coverage numbers in [0, 1]")
_arrs = list(PRED.values()) + list(SERIES.values()) + list(ABL.values()) + list(AVG.values())
need(all(a.shape == (R, N) for a in _arrs) and N == EXPECTED_ROWS and NC == EXPECTED_CELLS and len(cells) == 84, f"84 cells and {EXPECTED_ROWS:,} rows in every full-protocol result", f"{len(_arrs)} arrays")
_all_cells = set(cells) | set(PER_CELL.cell_id) | set(COVC_DF.cell_id) | set(EXT_CELLS.cell_id) | {c for f in ABL_FRAMES.values() for c in f.cell_id.unique()}
need(not (_all_cells & SENS_CELLS) and not any(str(c).startswith("b3") for c in _all_cells), "no batch-3 / sensitivity cells in any array, frame or table")
need(all(not set(s["train"]) & set(s["test"]) for s in SETS + EXT_SETS), "training members never include their own evaluation cells (all 18 outer sets and the extreme-holdout sets)",
     f"also asserted per member at training time ({MEMBER_AUDIT['members']} deep members, {MEMBER_AUDIT['trees']} forests this run) and on every cached frame by the test-cell coverage check")
need(all(OUT.resolve() in Path(p).resolve().parents for p in WRITTEN), "all outputs written under outputs/phase6b only", f"{len(WRITTEN)} files")
HASH1 = hash_state()
for _k in ("gate", "module", "verdict"):
    need(HASH1[_k]["sha"] == HASH0[_k]["sha"] == HASH1[_k]["exp"], f"sha256 unchanged at end: {_k}", HASH1[_k]["sha"][:16] + "...")
    need(HASH1[_k]["mtime"] == HASH0[_k]["mtime"], f"mtime unchanged at end: {_k}")
need(sha_raw(P_VERDICT) == VERDICT_SHA, "verdict file untouched (W300 FAIL / W240 PASS, never recomputed)")
TOTAL_INTERVALS = int(sum(INTERVALS.values()))
print("\nbootstrap intervals computed in this phase:", INTERVALS, "| total", TOTAL_INTERVALS)
chk("PASS", "total interval count printed", TOTAL_INTERVALS)

# ---- fixed-width report (style of phase6a_report.txt)
def tbl(df, cols, fm=None, hdr=None):
    fm = fm or {}; hdr = hdr or {}
    cell = lambda c, v: "-" if (isinstance(v, float) and np.isnan(v)) else (fm[c](v) if c in fm else (f"{v:.3f}" if isinstance(v, (float, np.floating)) else str(v)))
    rows = [[cell(c, v) for c, v in zip(cols, r)] for r in df[cols].itertuples(index=False, name=None)]; heads = [hdr.get(c, c) for c in cols]
    wd = [max(len(h), *(len(r[i]) for r in rows)) if rows else len(h) for i, h in enumerate(heads)]
    num = [pd.api.types.is_numeric_dtype(df[c]) and df[c].dtype != bool for c in cols]; fr_ = lambda r: "  ".join(x.rjust(wd[i]) if num[i] else x.ljust(wd[i]) for i, x in enumerate(r))
    return "\n".join([fr_(heads), "  ".join("-" * x for x in wd)] + [fr_(r) for r in rows])
pct = lambda v: f"{100 * v:.1f}%"; sg = lambda v: f"{v:+.3f}"
bar = "=" * 100; L = []; sec = lambda t: L.extend(["", bar, t, bar])
L += [bar, "PHASE 6b  ABLATIONS, ALERT RULES, EXTREME-LIFE HOLDOUT  (descriptive / exploratory; nothing here is a gate result)", bar,
      f"generated {dt.datetime.now():%Y-%m-%d %H:%M}   root {ROOT}   runtime {(time.time() - T_START) / 60:.1f} min   GPU {GPU_INFO}",
      f"gate sha256 {HASH1['gate']['sha']}", f"soh_deep_v2 sha256 (CRLF->LF) {HASH1['module']['sha']}", f"gate_verdict sha256 {HASH1['verdict']['sha']}  (frozen: W300 FAIL, W240 PASS; never recomputed)",
      f"frozen_config.json sha256 {FZ_SHA}", f"development rows {N:,} | cells {NC} | repeats {R} | err = pred - y (+ = over-estimated) | bootstrap: cells, {B_BOOT} draws, seed {SEED_BOOT}",
      f"total bootstrap intervals computed in this phase: {TOTAL_INTERVALS}  {INTERVALS}"]
sec("1. D69 FIXED CONFIG (M0, pooled inner MAE over all tuning-log rows) AND REPRODUCTION CHECK")
L.append(tbl(RANK_DF, ["window", "rank", "config", "lr", "wd", "pooled_inner_mae"], {"lr": lambda v: f"{v:g}", "wd": lambda v: f"{v:g}", "pooled_inner_mae": lambda v: f"{v:.6f}"}))
L += ["chosen: " + "; ".join(f"{w} {FROZEN_CFG[w]['config']} (B4r leaf {FROZEN_CFG[w]['b4r_leaf']})" for w in WINS), "", tbl(REPRO, ["window", "model", "recomputed", "deep_results", "abs_diff"], {"recomputed": lambda v: f"{v:.6f}", "deep_results": lambda v: f"{v:.6f}", "abs_diff": lambda v: f"{v:.1e}"})]
sec("2a. PER-CELL BIAS VS LIFE (Spearman, cell-bootstrap CI) AND LIFE TERTILES (cuts 493 / 710)")
L.append(tbl(SPEAR.assign(ci95=[f"[{a:+.3f}, {b:+.3f}]" for a, b in zip(SPEAR.ci95_lo, SPEAR.ci95_hi)]), ["window", "model", "subset", "n_cells", "spearman_bias_vs_life", "ci95"], {"spearman_bias_vs_life": sg}))
L += ["", tbl(TERT_BIAS, ["window", "model", "subset", "life_tertile", "n_cells", "mean_signed_err", "macro_mae", "share_cells_bias_pos"], {"mean_signed_err": sg, "share_cells_bias_pos": pct})]
sec("2b. JUNCTION ANALYSIS, STITCHED CELLS (step = 20 cycles after minus 20 before; error windows 100 cycles)")
L.append(tbl(pd.DataFrame([dict(cell_id=c, **v) for c, v in JUNC.items()]), ["cell_id", "j0", "j1", "step", "z", "used", "rule"], {"step": sg, "z": lambda v: f"{v:.1f}"}, {"j0": "approx", "j1": "refined", "step": "step_pp", "z": "x_median"}))
L += ["", tbl(JUNC_DF, ["window", "model", "cell_id", "true_step_pp", "pred_step_pp", "err_100_before", "err_100_after", "mean_err_pre_junction", "mean_err_all", "frac_of_mean_err_before", "mae_pre_junction", "mae_all"],
              {c: sg for c in ["true_step_pp", "pred_step_pp", "err_100_before", "err_100_after", "mean_err_pre_junction", "mean_err_all"]}, {"mean_err_pre_junction": "err_pre", "frac_of_mean_err_before": "frac_before", "mae_pre_junction": "mae_pre"})]
sec("2c. EXPLORATORY: per-row average of M0 ensemble and B4r (negative diff = average better; not a gate result)")
L.append(tbl(AVG_DF.assign(ci95=[f"[{a:+.3f}, {b:+.3f}]" for a, b in zip(AVG_DF.ci95_lo, AVG_DF.ci95_hi)]), ["window", "subset", "contrast", "n_cells", "avg_macro_mae", "other_macro_mae", "diff", "ci95"], {"diff": sg}))
sec("2d. ONE-SIDED 90% CONFORMAL LOWER BOUND (APPROXIMATE: calibration folds' models saw the test fold's cells)")
L.append(tbl(COV_DF.assign(ci95=[f"[{a:.3f}, {b:.3f}]" for a, b in zip(COV_DF.ci95_lo, COV_DF.ci95_hi)]), ["window", "model", "variant", "macro_cov", "ci95", "pooled_cov", "min_cell_cov", "share_cells_below_0p80", "mean_q_pp"], {"share_cells_below_0p80": pct}))
L += ["", tbl(COVG_DF.assign(ci95=[f"[{a:.3f}, {b:.3f}]" for a, b in zip(COVG_DF.ci95_lo, COVG_DF.ci95_hi)]), ["window", "model", "variant", "group_type", "group", "n_cells", "macro_cov", "ci95"])]
L += ["", "ALERT RULES: A = estimate <= threshold, B = lower bound <= threshold. delay = estimated - true crossing (negative = early); early alarm = more than 5% of life early.",
      "Right-censoring: scoring stops at end of life, so detections that would come after the last scored row are counted as MISSED, not late. Delay statistics use detected events only.",
      "85% truth = first scored cycle with true SOH <= 85; 80% truth = cycle_life. EMA alpha and conformal q use the other folds only. Nothing is chosen on test cells.",
      tbl(ALERT_DF, ["window", "model", "variant", "threshold", "rule", "subset", "n_events", "missed_rate", "early_alarm_rate", "median_delay_cycles", "mean_abs_delay_cycles", "median_delay_pct_life", "mean_abs_delay_pct_life"],
          {"missed_rate": pct, "early_alarm_rate": pct, "median_delay_cycles": lambda v: f"{v:+.0f}", "mean_abs_delay_cycles": lambda v: f"{v:.0f}", "median_delay_pct_life": lambda v: f"{v:+.1f}", "mean_abs_delay_pct_life": lambda v: f"{v:.1f}"},
          {"median_delay_pct_life": "med_%life", "mean_abs_delay_pct_life": "mabs_%life", "mean_abs_delay_cycles": "mabs_cyc", "median_delay_cycles": "med_cyc"})]
sec(f"3. TRAINING ABLATIONS (DESCRIPTIVE; {INTERVALS.get('3 paired ablation intervals', 0)} paired intervals; some will exclude zero by chance)")
L.append(tbl(ABL_DF.assign(ci95=[("" if r.base == "" else f"[{r.ci95_lo:+.3f}, {r.ci95_hi:+.3f}]") for r in ABL_DF.itertuples()]), ["window", "variant", "base", "macro_mae", "diff_vs_base", "ci95", "macro_excl_stitched", "last20_macro_mae", "pooled_bias"], {"diff_vs_base": sg, "pooled_bias": sg}))
L += ["", "SKIPPED variants:", tbl(pd.DataFrame(SKIPPED), ["variant", "status", "reason"]),
      "mse = Huber delta 1e6 pp (equals 0.5*x^2 for |x| <= delta, the same gradient scale as Huber's quadratic zone). B4r:+log_cycle is run on the tree model B4r (fixed leaf by the D69-style rule, no nested tuning), not on M0."]
sec(f"4. EXTREME-LIFE HOLDOUT (mode: {EXTREME_MODE}; policy twins of held-out cells removed from training)")
L.append(tbl(EXT_INFO_DF, ["group", "n_held", "n_policies_held", "n_twins_removed", "n_train", "train_life_min", "train_life_max"], {"train_life_min": lambda v: f"{v:.0f}", "train_life_max": lambda v: f"{v:.0f}"}))
L += ["", tbl(EXT_SUM, ["window", "group", "n_held", "mean_life", "mean_dist_from_train", "max_dist_from_train", "M0_extreme", "M0_D5b", "M0_Dref", "B4r_extreme", "B4r_D5b", "B4r_Dref", "M0_ratio_vs_D5b", "B4r_ratio_vs_D5b"],
              {"mean_life": lambda v: f"{v:.0f}", "mean_dist_from_train": lambda v: f"{v:.0f}", "max_dist_from_train": lambda v: f"{v:.0f}"})]
L += ["", tbl(EXT_CELLS, ["window", "group", "cell_id", "stitched", "life", "dist_from_train_range", "side", "M0_extreme_mae", "M0_D5b_mae", "B4r_extreme_mae", "B4r_D5b_mae"], {"life": lambda v: f"{v:.0f}", "dist_from_train_range": lambda v: f"{v:.0f}"})]
sec("5. CHECKS")
_cnt = {lv: sum(c["level"] == lv for c in CHECKS) for lv in ("PASS", "WARN", "FAIL")}; L.append(f"so far: {_cnt}")
L += [f"  [{c['level']}] {c['name']} | {c['detail']}" for c in CHECKS if c["level"] in ("WARN", "FAIL")]
_rp = save_text("\n".join(L) + "\n", "phase6b_report.txt")
def artifact_list():
    return [dict(file=str(p.relative_to(OUT)).replace("\\", "/"), bytes=p.stat().st_size) for p in sorted(OUT.rglob("*")) if p.is_file() and "cache" not in p.relative_to(OUT).parts]
dump_checks(partial=False, extra=dict(artifacts=artifact_list(), elapsed_seconds=round(time.time() - T_START, 1), hashes=dict(start=HASH0, end=HASH1), frozen_config_sha256=FZ_SHA, gpu=GPU_INFO))
keep_awake(False)
print(_rp.read_text(encoding="utf-8"))
print(f"\nFINAL CHECKS: PASS {sum(c['level'] == 'PASS' for c in CHECKS)} | WARN {sum(c['level'] == 'WARN' for c in CHECKS)} | FAIL {sum(c['level'] == 'FAIL' for c in CHECKS)} | intervals {TOTAL_INTERVALS}")
print("\nARTIFACTS (outputs/phase6b, cache excluded)")
print(pd.DataFrame(artifact_list()).assign(KB=lambda d: (d.bytes / 1024).round(1)).drop(columns="bytes").to_string(index=False))

[PASS] no NaN in headline table: per_cell_bias | 0 NaN cells
[PASS] no NaN in headline table: bias_vs_life_spearman | 0 NaN cells
[PASS] no NaN in headline table: bias_by_life_tertile | 0 NaN cells
[PASS] no NaN in headline table: ensemble_avg_M0_B4r | 0 NaN cells
[PASS] no NaN in headline table: onesided_coverage | 0 NaN cells
[PASS] no NaN in headline table: onesided_coverage_by_group | 0 NaN cells
[PASS] no NaN in headline table: alert_rules (rates) | 0 NaN cells
[PASS] no NaN in headline table: ablation_results | 0 NaN cells
[PASS] no NaN in headline table: ablation_results (paired) | 0 NaN cells
[PASS] no NaN in headline table: extreme_holdout_summary | 0 NaN cells
[PASS] no NaN in headline table: extreme_holdout_cells | 0 NaN cells
[PASS] coverage numbers in [0, 1]
[PASS] 84 cells and 55,963 rows in every full-protocol result | 34 arrays
[PASS] no batch-3 / sensitivity cells in any array, frame or table
[PASS] training members never include their own evaluation cells (all 18 oute

: 